In [ ]:
from datetime import datetime
from getpass import getpass

# 本ノートブックは テスト手順-管理者機能-S3-機関ストレージ-SigV4回帰.ipynb の型を
# 複製し、既存データの移行互換(S-10)を見るものに作り替えたもの。
# 回帰ノートブックは変更していない。複製したセルには複製元のセル番号を
# コメントで残してある(例: 「回帰 NB セル [6] の複製」)。

# === この NB 固有のパラメータ ==============================================
# phase: 'a' 仕込み(develop 版) / 'b' 検証(対象 PR 版) / 'c' ロールバック(develop 版)。
# None のままなら最初のセルで input() で聞く。papermill では必ず与えること。
phase = None
# 段をまたぐ控え。None なら evidence_dir/S-10-state.json。
# phase a が書き、b / c が読む。**資格情報は書かない。**
state_path = None
# ドット入りバケット(名前にドットを 2 つ入れる)を作るか。任意シナリオ S-10b-10。
run_dotted = True
# 200MB のマルチパートを含めるか。
run_chunked = True
# ===========================================================================

# URL は .config.yaml(admin_rdm_url / rdm_url)か papermill で与える。次のセルの
# 補完は None のパラメータにしか効かないので、ここに既定値を書くと .config.yaml が
# 無視される。
admin_rdm_url = None
rdm_url = None
idp_name_1 = 'GakuNin RDM IdP'

idp_username_1 = None
idp_password_1 = None

# admin_idp_name を指定すると Embedded DS(IdP選択) 経由でログインする
admin_idp_name = None
admin_username = None
admin_password = None
# entityID 直接指定（Embedded DS バイパス用）
admin_idp_entity_id = None
default_result_path = None
close_on_fail = False
transition_timeout = 60000

# 機関ストレージ設定用パラメータ
target_organization = None

# Amazon S3 設定。資格情報は .config.yaml か実行時プロンプトで与える。
# **ここに値を書かない**。
s3_access_key = None
s3_secret_key = None

# バケット。phase a で生成し、**phase c の後始末まで消さない**。
# phase b / c は S-10-state.json から名前を読むので、ここは None のままでよい。
s3_bucket_mig = None          # 通常バケット。*-inst / *-addon / *-newconn で使う
s3_bucket_migver = None       # バージョニング有効。*-ver で使う
s3_bucket_dot = None          # ドット入り名。任意の *-dot で使う

bucket_prefix = 'e2e-sigv4'
bucket_suffix = None
# **phase a / b では強制的に False にする**(次のセル)。phase c の末尾でだけ
# cleanup_buckets() を呼んで消す。
delete_generated_buckets = False

s3_region_default = 'us-east-1'
s3_server_side_encryption = False

# 機関ストレージの表示名（ファイルツリーに表示される名前）
# 'NII Storage' / 'United States' は osf.io 側でファイルツリーのアイコンが
# 通常アイコンのままになり、既定ストレージと見分けが付きにくいので使わない。
institutional_storage_name = 'S3 E2E Migration'

# S3 アドオン(外部ストレージ)の表示名と ID
addon_storage_name = 'Amazon S3'
addon_storage_id = 's3'

# S-10a で置くファイル。名前そのものが検証対象(SigV2 → SigV4 でエンコード規則が
# 変わる文字を含める)。
f_ascii = 'S-10-ascii.txt'
f_japanese = 'S-10-日本語.txt'
f_symbols = 'S-10-日本語 名前+記号(1).txt'
f_space = 'S-10 space.txt'
f_nested = 'nested.txt'
f_raw_key = 'S-10-raw/a+b %25 c~d.txt'      # boto3 で直接置く(GRDM 外から置かれたキー)
f_ver = 'S-10-ver.txt'
f_chunked = 'S-10-200MB.bin'
folder_top = 'S-10-folder'
folder_sub = 'サブ'

# S-10b-8 で置く新データ(S-10c で読む)
f_new_japanese = 'S-10b-new-日本語.txt'
f_new_chunked = 'S-10b-new-200MB.bin'
f_new_ver = 'S-10b-ver.txt'
# S-10b-11 で置くファイル
f_newconn = 'S-10b-newconn.txt'

# 200MB。WB の CONTIGUOUS_UPLOAD_SIZE_LIMIT(既定 128,000,000)を超えないと
# マルチパートにならない。
chunked_file_size = 200000000
wb_chunk_size = 64000000
wb_contiguous_upload_size_limit = 128000000

# 証跡の保存先。既定は ./evidence/run-<YYYYmmdd-HHMMSS>(リポジトリ直下)
evidence_dir = None

# プロジェクト名プレフィックス。**phase a が決めて控えに書き、b / c は控えから読む**
# (phase ごとに作り直すと別のプロジェクトを見てしまう)。回帰 NB とは別の名前に
# するため末尾に -mig を付ける。
rdm_project_prefix = None

In [ ]:
# 回帰 NB セル [1] の複製。
# .config.yaml があれば、未設定(None)のパラメータを同名キーで補完する
# （papermill実行時は先にパラメータが注入されるため、この処理は上書きしない）
import os as _os
import yaml as _yaml

_cfg_path = '.config.yaml'
if _os.path.exists(_cfg_path):
    with open(_cfg_path) as _f:
        _cfg = _yaml.safe_load(_f) or {}
    _loaded = []
    for _k, _v in _cfg.items():
        if _k in globals() and globals()[_k] is None and _v is not None:
            globals()[_k] = _v
            _loaded.append(_k)
    print(f'.config.yaml から補完: {_loaded}')
else:
    print('.config.yaml なし（プロンプト入力で続行）')

In [ ]:
# phase を確定し、段ごとの不変条件を立てる。
if phase is None:
    phase = input(prompt='phase (a=仕込み / b=検証 / c=ロールバック)')
phase = str(phase).strip().lower()
assert phase in ('a', 'b', 'c'), f'phase は a / b / c のいずれか: {phase!r}'

# phase a / b でバケットを消すと次の段で見るものが無くなる。パラメータで True が
# 来ても握りつぶさず、ここで落とす。
if phase in ('a', 'b') and delete_generated_buckets:
    print('[!] phase a / b では delete_generated_buckets を False に強制する'
          '(次の段でバケットを読むため)')
    delete_generated_buckets = False

_PHASE_NOTE = {
    'a': '仕込み。**試験環境は develop 版**であること',
    'b': '検証。**試験環境は対象 PR 版**であること。設定は一切変更しない',
    'c': 'ロールバック。**試験環境は develop 版**に戻してあること',
}
print(f'phase = {phase} — {_PHASE_NOTE[phase]}')
print(f'delete_generated_buckets = {delete_generated_buckets}')

In [ ]:
# 回帰 NB セル [2] / [3] の複製(バケット名を聞かない点も同じ)。
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if admin_username is None:
    admin_username = input(prompt='Admin Email (管理者画面ログイン用)')
if admin_password is None:
    admin_password = getpass(prompt=f'Password for {admin_username} (管理者画面)')
if s3_access_key is None:
    s3_access_key = input(prompt='S3 Access Key')
if s3_secret_key is None:
    s3_secret_key = getpass(prompt='S3 Secret Key')
(len(idp_username_1), len(idp_password_1), len(admin_username), len(admin_password),
 len(s3_access_key), len(s3_secret_key))

In [ ]:
# 回帰 NB セル [4] の複製(shutil を明示的に import する点だけ足した)。
import hashlib
import json
import os
import shutil
import tempfile

work_dir = tempfile.mkdtemp()

if evidence_dir is None:
    evidence_dir = os.environ.get('E2E_EVIDENCE_DIR', 'evidence')
run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
run_dir = os.path.abspath(os.path.join(evidence_dir, f'run-{run_id}'))
os.makedirs(run_dir, exist_ok=True)

# Playwright の成果物(video-N.webm / har.zip / console.log / last-*.png|html)は
# scripts/playwright.py が default_result_path 配下へ書く。既定のままだと
# work_dir に落ち、最終セルの `!rm -fr {work_dir}` ごと消える。har.zip は 302 の
# 判定に使うので、既定を run_dir にする。
if default_result_path is None:
    default_result_path = run_dir


def evidence(name):
    """証跡ファイルのパスを作る。名前には S-番号を含めること。"""
    return os.path.join(run_dir, name)


def record(name, text, quiet=False):
    """テキスト証跡を保存しつつ、ノートブックの出力にも残す。"""
    path = evidence(name)
    with open(path, 'w') as f:
        f.write(text)
    print(f'--- {path} ({len(text)} bytes)')
    if not quiet:
        print(text)
    return path


def record_json(name, obj, quiet=True):
    return record(name, json.dumps(obj, indent=2, ensure_ascii=False), quiet=quiet)


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


print(run_dir)
print(work_dir)

# GakuNinRDM 総合テスト [Amazon S3 — SigV4 移行互換 (S-10)]

- サブシステム名: 管理者 / ストレージ / WaterButler
- ページ/アドオン: 機関ストレージ (Amazon S3) および S3 アドオン
- 機能分類: 既存データの移行互換(上書きデプロイの可否判定)
- シナリオ名: **develop 版で作った設定とデータを、対象 PR 版がそのまま扱えるか**を
  確かめる。ついでに対象 PR 版で置いたデータを develop 版に戻して読めるかも見る
- **対象 PR: RDM-waterbutler#93(`feature/s3-sigv4`)/ RDM-osf.io#746**
- 取る証跡: `S-10a-*`(仕込み)/ `S-10b-*`(検証)/ `S-10c-*`(ロールバック)
- 用意するテストデータ: URL 一覧、アカウント(既存ユーザー1: GRDM)、管理者アカウント、
  AWS S3 のアクセスキー/シークレットキー。バケットは phase a が生成する

## このノートブックは 3 回に分けて実行する

**1 回の通し実行では成立しない。** 途中で試験環境のコードを入れ替える必要があるため、
`phase` パラメータを変えて 3 回実行する。

| 回 | `phase` | 実行前に試験環境を | 何をするか |
|---|---|---|---|
| 1 回目 | `'a'` | **develop 版**に切り替える | 旧実装で機関ストレージ登録・アドオン接続・データ投入を行い、メタデータとハッシュを控える |
| 2 回目 | `'b'` | **対象 PR 版**に切り替える | **設定を一切触らずに**同じものを開き、1 回目の控えと突合する。新データも置く |
| 3 回目 | `'c'` | **develop 版**に戻す | 2 回目が置いた新データを読み、後始末まで行う |

段をまたぐ控えは `S-10-state.json`(既定で証跡ディレクトリの下)。
**資格情報は控えに書かない。**

### phase 間で消してはいけないもの

- phase a が生成したバケット(`*-mig` / `*-migver` / 任意の dot 入り)
- phase a が作ったプロジェクト(`*-inst` / `*-addon` / `*-ver` / 任意の `*-dot`)
- `S-10-state.json`

`delete_generated_buckets` は phase a / b では**強制的に False** になる
(パラメータで True を渡しても落とす)。消すのは phase c の末尾だけ。

## phase b で設定を作り直してはいけない理由

phase b は「**develop 版が作った** `ExternalAccount` / `NodeSettings` / 機関ストレージ
設定を、対象 PR 版がそのまま読めるか」を見るためにある。
`register_institutional_s3()` や `connect_s3_addon()` を phase b で呼ぶと、
その設定が**対象 PR 版で作り直されて**しまい、確かめたかったことが消える。
phase b でこの 2 つを呼ぶのは S-10b-11(新規プロジェクトへの新規接続)だけで、
これは「新実装で新しく接続したものがどう保存されるか」を見る別の目的のもの。

## S-10 の判定項目

| 番号 | 何を見るか | 段 |
|---|---|---|
| S-10b-1 | 機関ストレージ設定が表示だけで読め、再保存を求められない | b |
| S-10b-2 | `*-inst` の一覧・ダウンロード(SHA-256 一致、302 → presigned) | b |
| S-10b-3 | `*-addon` の一覧(フォルダ展開、`+` ` ` `%25` `~` を含むキー) | b |
| S-10b-4 | `?meta=`(単体取得)と一覧の `etag` / `extra.md5` / `size` が phase a と一致。**ETag に引用符が残っていないこと** | b |
| S-10b-5 | 全ファイルのダウンロード SHA-256 一致 | b |
| S-10b-6 | intra move → copy | b |
| S-10b-7 | バージョン付きファイルの UI 削除で Versions / DeleteMarkers とも 0 | b |
| S-10b-8 | 新データ投入(S-10c 用) | b |
| S-10b-9 | 全段を通して WB 応答に 4xx/5xx と署名系の文言が無い | b |
| S-10b-10(任意) | ドット入りバケット名の一覧・ダウンロード | b |
| S-10b-11 | 新規プロジェクトに**新規接続**し、`folder_id` の保存形を記録 | b |
| S-10c-1 | 新データを develop 版で一覧・ダウンロード・版削除 | c |
| S-10c-2 | S-10b-11 で接続したプロジェクトを develop 版で開き、**何が起きるかを記録**(失敗が期待値。例外で止めない) | c |

## 機関ストレージの登録は「追加」ではなく「上書き」である

機関ストレージは機関あたり 1 つしかなく、phase a の登録は**現在の設定を破棄する**。
`S-10a-institutional-storage-before.txt` に登録前の状態が残るので、
phase c の後始末チェックリストで必ず元に戻すこと。**「登録を外す」操作は無い。**

In [ ]:
# 回帰 NB セル [6] の複製。役割を std/ver/apne1 から mig/migver/dot に替え、
# **phase をまたいで生かす**ために次の 2 点を変えてある。
#   - 名前はパラメータか S-10-state.json から来る(phase b / c は生成しない)
#   - delete_generated_buckets は phase c でしか True にならない(前のセル)
#
# 後始末は atexit に登録するので、以降のセルが例外で止まってもカーネル終了時に
# 走る。**ただし phase a / b では delete_generated_buckets=False なので消えない。**
import atexit
import random
import re
import time

from botocore.exceptions import ClientError

IAM_ACTIONS_NEEDED = (
    's3:CreateBucket, s3:PutBucketVersioning, s3:GetBucketVersioning, '
    's3:GetBucketLocation, s3:ListBucket, s3:ListBucketVersions, '
    's3:PutObject, s3:GetObject, s3:DeleteObject, s3:DeleteObjectVersion, '
    's3:AbortMultipartUpload, s3:ListMultipartUploadParts, s3:DeleteBucket')

generated_buckets = []
generated_bucket_names = []


def s3_client(region):
    """boto3 の S3 クライアント。資格情報はノートブックのパラメータから取る。"""
    import boto3
    from botocore.config import Config
    return boto3.client(
        's3',
        region_name=region,
        aws_access_key_id=s3_access_key,
        aws_secret_access_key=s3_secret_key,
        config=Config(signature_version='s3v4', retries={'max_attempts': 5}),
    )


def _check_credentials():
    """CreateBucket の前に、資格情報が AWS に通ることだけを確かめる。

    sts:GetCallerIdentity は IAM 権限を必要としないので、「鍵が違う」のか
    「権限が足りない」のかを切り分けられる。ARN は証跡にも出力にも残さない。
    """
    import boto3
    _sts = boto3.client('sts', region_name=s3_region_default,
                        aws_access_key_id=s3_access_key,
                        aws_secret_access_key=s3_secret_key)
    try:
        _sts.get_caller_identity()
    except ClientError as e:
        raise AssertionError(
            'S3 の資格情報が AWS に通らない'
            f'({e.response["Error"]["Code"]})。'
            '.config.yaml の s3_access_key / s3_secret_key が **AWS** の'
            'ものか確認すること。') from None
    print('  資格情報 (sts:GetCallerIdentity): OK')


def _new_bucket_suffix():
    return '{}-{:04d}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'),
                              random.randrange(10000))


def _generated_name(role, suffix):
    """生成バケット名。S3 の命名規則(小文字・数字・ハイフン、3〜63 文字)に沿う。"""
    name = f'{bucket_prefix}-{suffix}-{role}'.lower()
    assert re.fullmatch(r'[a-z0-9][a-z0-9-]{1,61}[a-z0-9]', name), \
        f'S3 のバケット名として使えない: {name}'
    return name


def _dotted_name(suffix):
    """ドット入りの名前。仮想ホスト形式が TLS で通らない名前の代表。

    ドットを含む名前は上の正規表現を通らないので別に作る。S3 の規則では
    ドットは使えるが、`<bucket>.s3.<region>.amazonaws.com` の証明書が
    ワイルドカード 1 段ぶんしか無いため、SDK はパス形式に落ちる。
    """
    name = f'{bucket_prefix}.{suffix}.mig'.lower()
    assert re.fullmatch(r'[a-z0-9][a-z0-9.-]{1,61}[a-z0-9]', name), \
        f'S3 のバケット名として使えない: {name}'
    return name


def _create_bucket(name, region):
    """バケットを作る。戻り値 'created' / 'exists-own'。"""
    c = s3_client(region)
    kwargs = {'Bucket': name}
    # us-east-1 に CreateBucketConfiguration を付けると InvalidLocationConstraint
    # になる。付けてよいのは us-east-1 以外のときだけ。
    if region != 'us-east-1':
        kwargs['CreateBucketConfiguration'] = {'LocationConstraint': region}
    try:
        c.create_bucket(**kwargs)
        status = 'created'
    except ClientError as e:
        code_ = e.response['Error']['Code']
        if code_ == 'BucketAlreadyOwnedByYou':
            status = 'exists-own'
        elif code_ == 'BucketAlreadyExists':
            raise
        elif code_ in ('AccessDenied', 'AllAccessDisabled'):
            raise AssertionError(
                f'CreateBucket({name}) が {code_} で失敗した。'
                f'必要な IAM 権限: {IAM_ACTIONS_NEEDED}')
        else:
            raise
    c.get_waiter('bucket_exists').wait(Bucket=name)
    return status


def _enable_versioning(name, region):
    """バージョニングを有効にし、get_bucket_versioning で Enabled を確認する。"""
    c = s3_client(region)
    c.put_bucket_versioning(
        Bucket=name, VersioningConfiguration={'Status': 'Enabled'})
    for _ in range(10):
        if c.get_bucket_versioning(Bucket=name).get('Status') == 'Enabled':
            return 'Enabled'
        time.sleep(1)
    raise AssertionError(f'{name} のバージョニングが Enabled にならない')


def _purge_bucket(name, region):
    """中身(全バージョン・削除マーカー・進行中の multipart)を空にする。"""
    c = s3_client(region)
    deleted = 0
    kwargs = {'Bucket': name}
    while True:
        r = c.list_object_versions(**kwargs)
        objs = [{'Key': v['Key'], 'VersionId': v['VersionId']}
                for v in r.get('Versions', []) + r.get('DeleteMarkers', [])]
        for i in range(0, len(objs), 1000):
            c.delete_objects(Bucket=name, Delete={'Objects': objs[i:i + 1000]})
        deleted += len(objs)
        if not r.get('IsTruncated'):
            break
        kwargs['KeyMarker'] = r.get('NextKeyMarker')
        kwargs['VersionIdMarker'] = r.get('NextVersionIdMarker')

    aborted = 0
    kwargs = {'Bucket': name}
    while True:
        r = c.list_multipart_uploads(**kwargs)
        for u in r.get('Uploads', []):
            c.abort_multipart_upload(Bucket=name, Key=u['Key'],
                                     UploadId=u['UploadId'])
            aborted += 1
        if not r.get('IsTruncated'):
            break
        kwargs['KeyMarker'] = r.get('NextKeyMarker')
        kwargs['UploadIdMarker'] = r.get('NextUploadIdMarker')
    return deleted, aborted


def cleanup_buckets(verbose=True):
    """このノートブックが作ったバケットを中身ごと削除する。

    **phase a / b では delete_generated_buckets=False なので名前を出すだけ。**
    実際に消えるのは phase c の末尾で呼んだときだけ。
    """
    lines = []
    if not generated_buckets:
        if generated_bucket_names:
            lines.append(f'  生成した {len(generated_bucket_names)} 件は'
                         '既にすべて削除済み')
        else:
            lines.append('  この実行で生成したバケットは無い')
    elif not delete_generated_buckets:
        for name, region in generated_buckets:
            lines.append(f'  {name} ({region}): delete_generated_buckets=False の'
                         'ため残した(次の段で使う)')
    else:
        for name, region in list(generated_buckets):
            for attempt in (1, 2, 3):
                try:
                    n, a = _purge_bucket(name, region)
                    s3_client(region).delete_bucket(Bucket=name)
                    lines.append(
                        f'  {name} ({region}): {n} オブジェクト削除 / '
                        f'{a} multipart 中止 / delete_bucket OK(試行 {attempt})')
                    generated_buckets.remove((name, region))
                    break
                except ClientError as e:
                    code_ = e.response['Error']['Code']
                    if code_ in ('NoSuchBucket', '404'):
                        lines.append(f'  {name} ({region}): 既に無い')
                        generated_buckets.remove((name, region))
                        break
                    if code_ == 'BucketNotEmpty' and attempt < 3:
                        continue
                    lines.append(f'  {name} ({region}): {code_} で削除できなかった'
                                 ' — **手動で削除すること**')
                    break
    text = '\n'.join(lines)
    if verbose:
        print(text)
    return text


if not globals().get('_cleanup_registered'):
    atexit.register(cleanup_buckets, verbose=False)
    _cleanup_registered = True


def bucket_state(bucket, region, generated):
    """バケットの前提(所在・中身の有無・バージョニング)を採る。"""
    c = s3_client(region)
    listed = c.list_objects_v2(Bucket=bucket, MaxKeys=5)
    # GetBucketLocation は us-east-1 を None(旧 API では '')で返す。
    raw = c.get_bucket_location(Bucket=bucket).get('LocationConstraint')
    return {
        'bucket': bucket,
        'generated': generated,
        'declared_region': region,
        'GetBucketLocation_raw': raw,
        'GetBucketLocation': raw or 'us-east-1',
        'KeyCount': listed.get('KeyCount', 0),
        'sample_keys': [o['Key'] for o in listed.get('Contents', [])],
        'Versioning': c.get_bucket_versioning(Bucket=bucket).get(
            'Status', 'Disabled'),
    }


print('バケット操作のヘルパを定義した(生成はこの後のセル)')

In [ ]:
# S-10-state.json: 段をまたぐ控え。**資格情報は書かない。**
#
# phase a が書き、b / c が読む。証跡ディレクトリは実行ごとに変わる(run_id が
# 時刻)ので、控えは既定で**最初の phase a の証跡ディレクトリ**に置かれる。
# phase b / c では state_path をそのパスで与えること(与えないと今回の run_dir を
# 見にいって「無い」で止まる)。
if state_path is None:
    state_path = evidence('S-10-state.json')
state_path = os.path.abspath(state_path)


def save_state(state):
    state['updated_at'] = datetime.now().isoformat()
    text = json.dumps(state, indent=2, ensure_ascii=False)
    # 短い文字列は控えの中の無関係な箇所に偶然現れる(1 文字のダミーパスワードで
    # 必ず落ちる)。8 文字未満は突き合わせから外し、外したことを黙らずに出す。
    for secret in (s3_access_key, s3_secret_key, idp_password_1, admin_password):
        if not secret:
            continue
        if len(secret) < 8:
            print('[!] 8 文字未満の資格情報があるため、控えの混入チェックから'
                  '外した(偶然の一致で必ず落ちるため)')
            continue
        assert secret not in text, '控えに資格情報が混ざっている'
    os.makedirs(os.path.dirname(state_path), exist_ok=True)
    with open(state_path, 'w') as f:
        f.write(text)
    print(f'--- 控えを書いた: {state_path} ({len(text)} bytes)')
    return state


def load_state():
    if not os.path.exists(state_path):
        raise AssertionError(
            f'控えが無い: {state_path}\n'
            'phase a を先に実行し、その証跡ディレクトリの S-10-state.json を'
            ' state_path パラメータで指すこと')
    with open(state_path) as f:
        return json.load(f)


if phase == 'a':
    state = {
        'created_at': datetime.now().isoformat(),
        'phase_runs': {},
        'buckets': {},
        'projects': {},
        'files': {},        # 論理名 -> {'sha256':…, 'size':…, 'where':…}
        'expected': {},     # 200MB のパート数など
    }
    print('phase a: 控えを新規に作る')
else:
    state = load_state()
    print(f'phase {phase}: 控えを読んだ(作成 {state.get("created_at")})')
    print(json.dumps({k: state.get(k) for k in ('buckets', 'projects')},
                     indent=2, ensure_ascii=False))

state.setdefault('phase_runs', {})[phase] = {'run_id': run_id, 'run_dir': run_dir}

In [ ]:
# バケットの確定。
#   phase a: 生成して控えに書く
#   phase b / c: 控えから読むだけ(**作らない**)
_check_credentials()

_region_of = {'mig': s3_region_default, 'migver': s3_region_default,
              'dot': s3_region_default}
_used_by = {'mig': ['*-inst', '*-addon', '*-newconn'], 'migver': ['*-ver'],
            'dot': ['*-dot']}

if phase == 'a':
    if bucket_suffix is None:
        bucket_suffix = _new_bucket_suffix()
    _roles = ['mig', 'migver'] + (['dot'] if run_dotted else [])
    _given = {'mig': s3_bucket_mig, 'migver': s3_bucket_migver,
              'dot': s3_bucket_dot}
    _resolved = {}
    for _attempt in (1, 2):
        _resolved = {}
        try:
            for _role in _roles:
                _region = _region_of[_role]
                if _given[_role]:
                    _resolved[_role] = (_given[_role], _region, False)
                    print(f'  {_role}: {_given[_role]} ({_region}) -> 指定(削除しない)')
                    continue
                _name = (_dotted_name(bucket_suffix) if _role == 'dot'
                         else _generated_name(_role, bucket_suffix))
                _status = _create_bucket(_name, _region)
                generated_buckets.append((_name, _region))
                generated_bucket_names.append((_name, _region))
                _resolved[_role] = (_name, _region, True)
                print(f'  {_role}: {_name} ({_region}) -> {_status}')
            break
        except ClientError as _e:
            if _e.response['Error']['Code'] != 'BucketAlreadyExists' or _attempt == 2:
                raise
            print('BucketAlreadyExists(他人所有の名前と衝突)。'
                  'suffix を変えて 1 回だけ再試行する')
            # 作りかけは消す。delete_generated_buckets=False なので cleanup_buckets
            # では消えない。ここだけは直接消す。
            for _n, _r in list(generated_buckets):
                try:
                    s3_client(_r).delete_bucket(Bucket=_n)
                except ClientError:
                    pass
                generated_buckets.remove((_n, _r))
            generated_bucket_names.clear()
            bucket_suffix = _new_bucket_suffix()

    s3_bucket_mig = _resolved['mig'][0]
    s3_bucket_migver = _resolved['migver'][0]
    if _resolved['migver'][2]:
        _enable_versioning(s3_bucket_migver, _resolved['migver'][1])
    s3_bucket_dot = _resolved['dot'][0] if run_dotted else None

    state['buckets'] = {
        'prefix': bucket_prefix, 'suffix': bucket_suffix,
        'mig': s3_bucket_mig, 'migver': s3_bucket_migver, 'dot': s3_bucket_dot,
        'region': s3_region_default,
        'generated': [list(x) for x in generated_bucket_names],
    }
else:
    _b = state['buckets']
    s3_bucket_mig = _b['mig']
    s3_bucket_migver = _b['migver']
    s3_bucket_dot = _b.get('dot')
    bucket_prefix, bucket_suffix = _b['prefix'], _b['suffix']
    run_dotted = run_dotted and bool(s3_bucket_dot)
    # phase c の後始末で消すのは「phase a が生成した」ものだけ。控えから戻す。
    generated_buckets[:] = [tuple(x) for x in _b.get('generated', [])]
    generated_bucket_names[:] = list(generated_buckets)
    print(f'  控えから: mig={s3_bucket_mig} / migver={s3_bucket_migver} / '
          f'dot={s3_bucket_dot}')

_roles_now = ['mig', 'migver'] + (['dot'] if run_dotted else [])
_names_now = {'mig': s3_bucket_mig, 'migver': s3_bucket_migver,
              'dot': s3_bucket_dot}
_states = {}
for _r in _roles_now:
    _s = bucket_state(_names_now[_r], _region_of[_r],
                      any(_n == _names_now[_r] for _n, _rg in generated_buckets))
    _s['role'] = _r
    _s['used_by'] = _used_by[_r]
    assert _s['GetBucketLocation'] == _s['declared_region'], (
        f"{_s['bucket']} の所在は {_s['GetBucketLocation']} で、"
        f"宣言した {_s['declared_region']} と違う")
    _states[_r] = _s
assert _states['migver']['Versioning'] == 'Enabled', \
    f'{s3_bucket_migver} のバージョニングが有効でない(S-10b-7 の前提)'

record_json(f'S-10{phase}-buckets.json', {
    'phase': phase,
    'bucket_prefix': bucket_prefix,
    'bucket_suffix': bucket_suffix,
    'delete_generated_buckets': delete_generated_buckets,
    'buckets': [_states[_r] for _r in _roles_now],
    'note': ('phase a が生成し、phase c の後始末まで残す。'
             'GetBucketLocation_raw が null なのは us-east-1 のバケット。'),
}, quiet=False)

In [ ]:
# プロジェクト名。phase a が決めて控えに書き、b / c は控えから読む。
if phase == 'a':
    if rdm_project_prefix is None:
        rdm_project_prefix = 'TEST-S3-SIGV4-{}-mig'.format(
            datetime.now().strftime('%Y%m%d-%H%M%S'))
    state['projects'] = {
        'prefix': rdm_project_prefix,
        'inst': f'{rdm_project_prefix}-inst',
        'addon': f'{rdm_project_prefix}-addon',
        'ver': f'{rdm_project_prefix}-ver',
        'dot': f'{rdm_project_prefix}-dot' if run_dotted else None,
        # S-10b-11 で phase b が作る。名前だけ先に決めておく(phase c が読む)。
        'newconn': f'{rdm_project_prefix}-newconn',
    }
else:
    rdm_project_prefix = state['projects']['prefix']

p_inst = state['projects']['inst']
p_addon = state['projects']['addon']
p_ver = state['projects']['ver']
p_dot = state['projects'].get('dot')
p_newconn = state['projects']['newconn']
save_state(state)
print('\n'.join(f'  {k}: {v}' for k, v in state['projects'].items()))

In [ ]:
# 回帰 NB セル [7] の複製。
import asyncio
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

In [ ]:
# 回帰 NB セル [9] の複製。WB が返した HTTP ステータスと本文を捕まえる。
wb_responses = []

# 一覧応答(?meta=)は本文を解析するので切らない。それ以外は証跡が膨らむので切る。
WB_BODY_LIMIT = 2000
WB_BODY_LIMIT_META = 4 * 1024 * 1024

try:
    _recorder_attached
except NameError:
    _recorder_attached = []


async def _attach_recorder(page):
    if any(p is page for p in _recorder_attached):
        print('[i] 応答レコーダは登録済み(セルの再実行)。二重に付けない')
        return

    async def _on_response(response):
        url = response.url
        if '/v1/resources/' not in url and '/wb/' not in url:
            return
        entry = {
            'status': response.status,
            'method': response.request.method,
            'url': url,
            'at': datetime.now().isoformat(),
        }
        if response.request.method != 'GET':
            # action: "move" / "copy" は *要求* 本文にしかない。
            entry['request_body'] = (response.request.post_data or '')[:WB_BODY_LIMIT]
        limit = WB_BODY_LIMIT_META if 'meta=' in url else WB_BODY_LIMIT
        try:
            entry['body'] = (await response.text())[:limit]
        except Exception as e:
            entry['body'] = f'<unavailable: {e!r}>'
        wb_responses.append(entry)

    page.on('response', lambda r: asyncio.ensure_future(_on_response(r)))
    _recorder_attached.append(page)


async def _step(page):
    await _attach_recorder(page)

await run_pw(_step)


def _redact(text):
    """署名と資格情報を伏せる。証跡に出す前に必ず通す。"""
    import re
    out = text
    for secret in [s3_secret_key, s3_access_key]:
        if secret:
            out = out.replace(secret, '***')
    out = re.sub(r'(X-Amz-(?:Signature|Credential|Security-Token)=)[^&\s"\\]+',
                 r'\1***', out)
    out = re.sub(r'AKIA[0-9A-Z]{12,}', 'AKIA***', out)
    # 一覧ツールバーのダウンロード URL には OSF の accessToken が `?token=` で乗る。
    out = re.sub(r'([?&]token=)[^&\s"\\]+', r'\1***', out)
    return out


def dump_wb_responses(name, keep=False, quiet=True):
    """直近の操作で観測した WB 応答を証跡に落とす。"""
    text = _redact(json.dumps(wb_responses, indent=2, ensure_ascii=False))
    assert 'AKIA' not in text.replace('AKIA***', ''), '証跡にアクセスキーが残っている'
    path = record(name, text, quiet=quiet)
    if not keep:
        wb_responses.clear()
    return path


def wb_failures(entries=None):
    """4xx / 5xx と、署名系の文言を含む応答を拾う。"""
    entries = wb_responses if entries is None else entries
    bad_words = ('SignatureDoesNotMatch', 'AuthorizationHeaderMalformed',
                 'PermanentRedirect', 'InvalidAccessKeyId', 'InvalidRequest')
    out = []
    for e in entries:
        body = e.get('body') or ''
        if e['status'] >= 400 or any(w in body for w in bad_words):
            out.append(e)
    return out


# 全段を通した観測(S-10b-9 / S-10c)。dump_wb_responses で消える前に溜める。
wb_all = []


def keep_all(name):
    """証跡に落としつつ、全段ぶんの控えにも足す。"""
    wb_all.extend(wb_responses)
    return dump_wb_responses(name)

In [ ]:
# 回帰 NB セル [9] 後半の複製(ストレージノードを開く・growl・アップロード待ち)。
async def open_storage(page, storage_name):
    """ファイルツリーのストレージノードを開き、アップロードボタンが出る状態にする。"""
    node = page.locator(
        '//*[contains(@class, "title-text")'
        f' and starts-with(normalize-space(.), "{storage_name}")]').first
    await expect(node).to_be_visible(timeout=transition_timeout)
    await node.click(timeout=transition_timeout)
    await asyncio.sleep(2)
    upload_btn = page.locator(
        '//i[contains(@class, "fa-upload")]/../*[text() = "アップロード"]')
    await expect(upload_btn).to_be_visible(timeout=transition_timeout)


# ファイルツリーのノードのアイコンは osf.io の
# website/project/views/file.py の grid_data._inner が決めている。機関ストレージ
# (provider が osfstorage で nodeRegion を持つノード)は、ストレージ名が
# 'NII Storage' / 'United States' のときだけ通常アイコン
# (/static/addons/osfstorage/comicon.png)のままで、それ以外の名前のときに
# comicon_custom_storage.png に差し替わる。つまりアイコンの種類はストレージ名
# 次第なので、どちらでも拾えるように「/static/addons/osfstorage/comicon」で
# 始まる背景画像を目印にする。
INSTITUTIONAL_STORAGE_ICON = '/static/addons/osfstorage/comicon'

INSTITUTIONAL_STORAGE_TITLE_XPATH = (
    '//*[contains(@class, "tb-td-first")]'
    f'//*[contains(@style, "{INSTITUTIONAL_STORAGE_ICON}")]'
    '/../../following-sibling::*[contains(@class, "title-text")]')

NO_INSTITUTIONAL_STORAGE_NODE = (
    'ファイルツリーに機関ストレージのノードが見つからない。ノードのアイコンは'
    'osf.io 側がストレージ名から決めており、名前が "NII Storage" / '
    '"United States" のときは通常アイコン'
    '(/static/addons/osfstorage/comicon.png)、それ以外の名前のときは'
    '/static/addons/osfstorage/comicon_custom_storage.png になる。ここは'
    '両方を見ているので、見つからないのはアイコンの違いではなく、この'
    'プロジェクトに機関ストレージが付いていない可能性が高い。')


async def resolve_institutional_storage_name(page):
    """ファイルツリーの機関ストレージノードの表示名を読む。

    表示名は管理画面の #storage_name の値で決まるので環境ごとに違う。
    アイコンは通常・custom のどちらもありうるので両方を拾い、候補が複数
    あればパラメータ institutional_storage_name と一致するものを優先する。
    """
    titles = page.locator(INSTITUTIONAL_STORAGE_TITLE_XPATH)
    try:
        await expect(titles.first).to_be_visible(timeout=transition_timeout)
    except AssertionError:
        raise AssertionError(NO_INSTITUTIONAL_STORAGE_NODE)
    names = [n.strip() for n in await titles.all_inner_texts() if n.strip()]
    if not names:
        raise AssertionError(NO_INSTITUTIONAL_STORAGE_NODE)
    if len(names) > 1:
        print(f'[!] 機関ストレージ候補のノードが {len(names)} 件ある: {names}')
        if institutional_storage_name in names:
            print('    -> パラメータと一致する '
                  f'{institutional_storage_name!r} を使う')
            return institutional_storage_name
        print(f'    -> 一致するものが無いので先頭の {names[0]!r} を使う')
    return names[0]


async def open_institutional_storage(page):
    """機関ストレージのノードを開く。表示名は実際のノードから取る。"""
    global institutional_storage_name
    name = await resolve_institutional_storage_name(page)
    if name != institutional_storage_name:
        print('[!] 機関ストレージの表示名がパラメータと違う: '
              f'パラメータ={institutional_storage_name!r} / 実際={name!r}'
              ' -> 実際の表示名で進める')
        institutional_storage_name = name
    await open_storage(page, name)


async def clear_growls(page):
    """残っている growl を消す。判定の直前に呼ぶ。"""
    await page.evaluate(
        '() => { document.querySelectorAll(\'[data-growl="container"]\')'
        '.forEach(function(e) { e.remove(); }); }')


async def growl_messages(page):
    """今表示されている growl の本文。判定の根拠として証跡に残す。"""
    return await page.locator('//*[@data-growl = "message"]').all_inner_texts()


async def uploading_rows(page, filename=None):
    """アップロード中(progressbar 付きの仮行)の数。"""
    if filename is None:
        return await page.locator('//*[@role = "progressbar"]').count()
    return await page.locator(
        f'//*[text() = "{filename}"]/../following-sibling::*//*[@role = "progressbar"]'
    ).count()


async def wait_for_upload_finished(page, filename, timeout=None):
    """成否を問わず「アップロードが終わった」ことを待ち、経過秒を返す。

    fangorn は成功時も失敗時も仮行を消す。**仮行が残っている = XHR がまだ
    返っていない。** Dropzone は parallelUploads: 1 なので、終わっていない行を
    残したまま次へ進むと次のアップロードはキューで止まる。
    """
    timeout = transition_timeout * 10 if timeout is None else timeout
    started = time.time()
    deadline = started + timeout / 1000
    while time.time() < deadline:
        if await uploading_rows(page, filename) == 0:
            return time.time() - started
        await asyncio.sleep(2)
    raise AssertionError(
        f'{filename} のアップロードが {timeout / 1000:.0f} 秒で終わらない')


async def upload_and_wait(page, path):
    """アップロードして、行が出るまで待つ。"""
    name = os.path.basename(path)
    await grdm.upload_file(page, path)
    await wait_for_upload_finished(page, name,
                                   timeout=transition_timeout * 60)
    await expect(grdm.get_select_file_title_locator(page, name)
                 ).to_be_visible(timeout=transition_timeout * 3)
    return name

In [ ]:
# 回帰 NB セル [15] の複製。
import urllib.parse


async def admin_login(page):
    await page.goto(admin_rdm_url)
    email_field = page.locator('#id_email')
    django_form = await email_field.count() > 0 and await email_field.is_visible()
    if django_form:
        # Django メール/パスワードフォーム（enable_form=True の環境）
        await email_field.fill(admin_username)
        await page.locator('#id_password').fill(admin_password)
        await page.locator('//form//button[@type="submit"]').click()
    elif admin_idp_entity_id is not None:
        # entityID 直接指定で Shibboleth ログインを開始（Embedded DS バイパス）
        base = admin_rdm_url.rstrip('/')
        url = (base + '/Shibboleth.sso/Login'
               '?entityID=' + urllib.parse.quote(admin_idp_entity_id, safe='')
               + '&target=' + urllib.parse.quote(base + '/account/shib-login', safe=''))
        await page.goto(url)
        print('[1] after goto:', page.url)
        title = await page.title()
        assert 'Unknown Identity Provider' not in title, (
            f'SPが entityID を認識していません: {admin_idp_entity_id}')
    else:
        # GakuNin Embedded DS（「所属している機関を選択」）から IdP を選ぶ。
        # embedded-wayf.js の submitForm() は #keytext を IdP 名と突き合わせ、
        # hidden の #user_idp に entityID を入れてから送信する。
        assert admin_idp_name is not None, (
            '管理者ログインがメール/パスワード形式ではない（Embedded DS が出ている）。'
            '.config.yaml に admin_idp_name を書くか、admin_idp_entity_id を'
            '直接指定すること。')
        _js = ('(name) => { const n = name.toLowerCase();'
               ' for (const e of (window.inc_search_list || []))'
               '   for (let j = 3; j < e.length; j++)'
               '     if (String(e[j]).toLowerCase() === n) return e[0];'
               ' return null; }')
        entity_id = await page.evaluate(_js, admin_idp_name)
        assert entity_id, (
            f'Embedded DS の IdP 一覧に {admin_idp_name!r} が無い。')
        print(f'[1] Embedded DS: {admin_idp_name} -> entityID={entity_id}')
        keytext = page.locator('#keytext')
        await expect(keytext).to_be_editable(timeout=transition_timeout)
        await keytext.click()
        await keytext.fill('')
        # incsearch(suggest.js)の［選択］は keyEvent -> checkLoop -> search の
        # 順でしか disabled が外れず、checkLoop のタイマは**キーイベントでしか
        # 起動しない**。fill() は input イベントしか出さないので 1 文字ずつ打つ。
        await keytext.type(admin_idp_name, delay=50)
        wayf_submit = page.locator('#wayf_submit_button')
        await expect(wayf_submit).to_be_enabled(timeout=transition_timeout)
        await wayf_submit.click()

    if not django_form:
        username = page.locator('#username')
        j_username = page.locator('input[name="j_username"]')
        consent = page.locator('#_shib_idp_doNotRememberConsent')
        logout_link = page.locator('//*[@href="/account/logout/"]')
        await expect(username.or_(j_username).or_(consent).or_(logout_link).first
                     ).to_be_visible(timeout=transition_timeout)
        if await username.count() > 0 and await username.is_visible():
            await username.fill(admin_username)
            await page.locator('#password').fill(admin_password)
            await page.locator(
                '//button[@type="submit"] | //input[@type="submit"]').first.click()
        elif await j_username.count() > 0 and await j_username.is_visible():
            await j_username.fill(admin_username)
            await page.locator('input[name="j_password"]').fill(admin_password)
            await page.locator(
                '//button[@type="submit"] | //input[@type="submit"]').first.click()
        else:
            print('[2] ログインフォームなし（既ログイン or 同意画面）:', page.url)

        await asyncio.sleep(2)
        err = page.locator('.form-error, .output--error, p.form-element.form-error')
        if await err.count() > 0 and await err.first.is_visible():
            raise AssertionError('IdPログイン失敗: '
                                 + (await err.first.text_content()).strip())
        try:
            await expect(consent).to_be_visible(timeout=15000)
            await consent.click()
            proceed = page.locator('//*[@name="_eventId_proceed"]')
            await expect(proceed).to_be_enabled()
            await proceed.click()
            print('[3] consent done:', page.url)
        except AssertionError:
            print('[3] 同意画面はスキップ:', page.url)

    logout_btn = page.locator(
        '//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link2 = page.locator('//*[@href="/account/logout/"]')
    try:
        await expect(logout_btn.or_(logout_link2)).to_be_visible(
            timeout=transition_timeout)
    except AssertionError:
        print('[4] 最終確認失敗。現在地:', page.url, '/', await page.title())
        print(await page.evaluate(
            '() => document.body.innerText.substring(0, 800)'))
        raise
    print('[4] 管理者画面ログイン成功:', page.url)


async def admin_logout(page):
    """回帰 NB セル [22] の複製。"""
    logout_btn = page.locator(
        '//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    if await logout_btn.count() > 0:
        await logout_btn.click()
    else:
        await logout_link.click()
    await expect(page.locator('.login-logo')).to_be_visible(
        timeout=transition_timeout)

In [ ]:
# 回帰 NB セル [17] の複製。
async def goto_institutional_storage(page):
    await page.goto(admin_rdm_url.rstrip('/')
                    + '/custom_storage_location/institutional_storage/')
    # サイドメニューの「機関ストレージ」(「機関ストレージのクォータ」と区別)
    link = page.locator(
        '//a[contains(@href, "institutional_storage")'
        ' and not(contains(@href, "quota"))]').first
    await expect(link).to_be_visible(timeout=transition_timeout)
    await link.click()
    await expect(page.locator('//input[@type="radio"]').first
                 ).to_be_visible(timeout=transition_timeout)


async def select_institution(page):
    """機関のリストから target_organization を選ぶ。

    単一機関の環境では機関リストが出ず、いきなり設定画面になる。
    """
    if target_organization is None:
        print('target_organization 未指定。機関リストの選択はスキップ')
        return
    link = page.locator(f'//a[text() = "{target_organization}"]')
    if await link.count() == 0:
        print(f'機関リストに "{target_organization}" が見つからない。'
              '既に設定画面とみなす')
        return
    await link.first.click()
    await expect(page.locator('//h2[contains(text(), "Institutional Storage")]')
                 ).to_be_visible(timeout=transition_timeout)


async def capture_institutional_storage_state(page, name):
    """選択状態と表示名を証跡に残す。後始末で戻すときの手がかりになる。"""
    state_ = await page.evaluate(
        '() => { var r = document.querySelector("input[name=options]:checked");'
        ' var n = document.getElementById("storage_name");'
        ' var b = document.getElementById("s3_bucket");'
        ' return {selected: r ? r.value : "none",'
        '         storage_name: n ? n.value : null,'
        '         s3_bucket_field: b ? b.value : null}; }')
    await page.screenshot(path=evidence(name + '.png'), full_page=True)
    record(name + '.txt', json.dumps(state_, indent=2, ensure_ascii=False))
    return state_

In [ ]:
# 回帰 NB セル [18] の複製。**phase a でしか呼ばない。**
async def register_institutional_s3(page, bucket, label):
    """Amazon S3 を機関ストレージとして登録する。**既存設定の上書きである。**"""
    await goto_institutional_storage(page)
    await select_institution(page)

    radio = page.locator('//input[@type="radio" and @value="s3"]')
    await expect(radio).to_be_visible(timeout=transition_timeout)
    await radio.scroll_into_view_if_needed()
    try:
        await radio.check()
    except Exception:
        await radio.check(force=True)
    await expect(radio).to_be_checked()
    selected = await page.evaluate(
        '() => { const r = document.querySelector("input[name=options]:checked");'
        ' return r ? r.value : "none"; }')
    assert selected == 's3', f'Amazon S3 を選択できていません: {selected}'

    storage_name = page.locator('#storage_name')
    sn_visible = await storage_name.is_visible()
    sn_enabled = await storage_name.is_enabled() if sn_visible else False
    sn_value = await storage_name.input_value() if sn_visible else ''
    print(f'storage_name: visible={sn_visible}, enabled={sn_enabled}, '
          f'value="{sn_value}"')
    if sn_visible and sn_enabled and not sn_value.strip():
        await storage_name.fill(institutional_storage_name)
    else:
        why = ('入力欄が出ていない' if not sn_visible
               else '入力欄が無効' if not sn_enabled
               else '既に名前が入っている')
        print(f'    ストレージ名は入力しない({why})。'
              f'既存の名前 "{sn_value}" のまま進める。'
              'ノードはファイルツリーのアイコンで特定するので、'
              '名前がパラメータと違っても見つかる。')

    save_btn = page.locator(
        '//button[@type="submit" and contains(@class, "btn-success")]').first
    await save_btn.scroll_into_view_if_needed()
    # submit ハンドラは rdm-institutional-storage-page.js が bind する。
    # ラジオボタンは HTML の解析時点で見えるので、見えた直後に押すとハンドラが
    # まだ無く、素のフォーム送信(GET)でページが再読込みされるだけになる。
    await page.wait_for_function(
        '() => { var f = document.getElementById("institutional_storage_form");'
        ' if (!f || !window.jQuery) { return false; }'
        ' var ev = window.jQuery._data(f, "events");'
        ' return !!(ev && ev.submit && ev.submit.length); }',
        timeout=transition_timeout)
    await save_btn.click()

    confirm_text = page.locator('#bbConfirmText')
    s3_modal = page.locator('#s3_modal')
    # 確認ダイアログ(bootbox)は osfstorage 以外では必ず出る。出ないのは送信が
    # 成立していないということなので、黙って先へ進めない。
    try:
        await expect(confirm_text).to_be_visible(timeout=10000)
    except Exception:
        raise AssertionError(
            '保存の確認ダイアログが出ない。フォーム送信が成立していない'
            '(submit ハンドラ未 bind、または checkValidity 失敗)。')
    confirm_strong = page.locator(
        '//div[contains(@class, "bootbox-body")]//strong')
    confirmation_string = await confirm_strong.first.text_content()
    print(f'Confirmation string: {confirmation_string}')
    await confirm_text.fill(confirmation_string)
    await page.locator(
        '//div[contains(@class, "modal") and contains(@class, "bootbox")]'
        '//button[contains(@class, "btn-danger")]').click()
    await asyncio.sleep(1)

    # 確認文字列が合わないと callback は呼ばれず、ダイアログだけ閉じて
    # 「Verification failed」の growl が残る。growl は自動では消えない。
    verify_failed = page.locator(
        '//*[contains(@class, "alert")][contains(., "Verification failed")]')
    assert await verify_failed.count() == 0, \
        f'確認文字列の照合に失敗した: "{confirmation_string}"'
    print('確認ダイアログを処理しました')

    await expect(s3_modal).to_be_visible(timeout=20000)
    await page.locator('#s3_access_key').fill(s3_access_key)
    await page.locator('#s3_secret_key').fill(s3_secret_key)
    await page.locator('#s3_bucket').fill(bucket)
    await page.evaluate(
        '() => { document.querySelectorAll("#s3_modal input")'
        '.forEach(el => el.dispatchEvent(new Event("keyup", { bubbles: true }))); }')
    await asyncio.sleep(1)

    connect_btn = page.locator('#s3_connect')
    await expect(connect_btn).to_be_enabled(timeout=transition_timeout)
    await connect_btn.click()
    save_modal_btn = page.locator('#s3_save')
    await expect(save_modal_btn).to_be_enabled(timeout=transition_timeout)

    msg = await page.evaluate(
        '() => document.getElementById("s3_message")'
        ' ? document.getElementById("s3_message").innerText : "no message element"')
    print('server message:', msg)
    record(f'{label}-connect-message.txt',
           f'bucket={bucket}\nserver message={msg}\n')

    await save_modal_btn.click()
    await expect(s3_modal).to_be_hidden(timeout=transition_timeout)
    await asyncio.sleep(2)

    # モーダルを閉じただけでは、ページは保存前に描画されたままである。
    # サーバから読み直してから選択状態と表示名を採る。
    await goto_institutional_storage(page)
    await select_institution(page)
    state_ = await capture_institutional_storage_state(
        page, f'{label}-institutional-storage-after')
    assert state_['selected'] == 's3', (
        f'保存後に選択されているのが Amazon S3 ではない: {state_["selected"]}')
    print(f'{label}: {bucket} を機関ストレージとして登録した')

In [ ]:
# 回帰 NB セル [24] 〜 [26] の複製。
import scripts.grdm
importlib.reload(scripts.grdm)


async def user_login(page):
    await page.goto(rdm_url)
    await page.wait_for_load_state('networkidle')
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count() > 0 and await consent_button.is_visible():
        await consent_button.click()
        await page.wait_for_load_state('networkidle')
    await scripts.grdm.expect_anonymous_toppage(
        page, idp_name_1, transition_timeout=transition_timeout)
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1,
        transition_timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-create-project-modal-button]')
                 ).to_be_visible(timeout=transition_timeout)


async def wait_dashboard_loaded(page, retries=3):
    """ダッシュボードのプロジェクト一覧が読み終わるのを待つ。

    [新規プロジェクト作成] ボタンは一覧より先に出るので、それだけを待つと
    「読み込み中」と「プロジェクトが無い」を区別できない。読み込み中のまま
    ensure_project_exists を呼ぶと「無い」と誤判定して同名プロジェクトを
    二重に作ってしまうので、必ずこれを先に通すこと。
    """
    loaded = page.locator(
        '//*[@data-test-dashboard-item-title]'
        ' | //*[contains(text(), "まだプロジェクトがありません")]').first
    create_btn = page.locator('//*[@data-test-create-project-modal-button]')
    no_api = page.locator('//*[@data-analytics-scope = "No api page"]')
    for attempt in range(1, retries + 1):
        reason = None
        try:
            await expect(create_btn.or_(no_api).first
                         ).to_be_visible(timeout=transition_timeout)
            if await no_api.count() > 0:
                reason = 'Ember が「APIは利用できません」を出している'
            else:
                await expect(loaded).to_be_visible(timeout=transition_timeout)
                return
        except AssertionError:
            reason = 'プロジェクト一覧がスピナーのまま'
        if attempt == retries:
            raise AssertionError(
                f'ダッシュボードが読み込まれない({reason})。'
                f' 現在地: {page.url}') from None
        print(f'[!] {reason}。再読込する ({attempt}/{retries - 1})')
        await asyncio.sleep(30)
        await page.goto(rdm_url)


async def open_project(page, project_name):
    """ダッシュボードから対象プロジェクトを開いてファイルタブを表示する。"""
    await page.goto(rdm_url)
    await wait_dashboard_loaded(page)
    await page.locator(
        f'//*[@data-test-dashboard-item-title and text()="{project_name}"]'
    ).first.click(timeout=transition_timeout)
    await expect(page.locator('#projectNavFiles')).to_be_visible(
        timeout=transition_timeout)


async def open_files_tab(page):
    await page.locator('#projectNavFiles a').click()
    await expect(page.locator('//*[@id = "treeGrid"]')).to_be_visible(
        timeout=transition_timeout)
    await asyncio.sleep(3)


async def create_project_and_open_files(page, project_name):
    await page.goto(rdm_url)
    await wait_dashboard_loaded(page)
    await scripts.grdm.ensure_project_exists(page, project_name,
                                             transition_timeout)
    await open_project(page, project_name)
    await open_files_tab(page)


async def open_project_files(page, project_name):
    """既にあるプロジェクトを開いてファイルタブを出す(作らない)。"""
    await open_project(page, project_name)
    await open_files_tab(page)

In [ ]:
# 回帰 NB セル [39] の複製。
# **phase b で呼んでよいのは S-10b-11(新規プロジェクト)だけ。**
# 既存プロジェクトに対して呼ぶと、develop 版が作った NodeSettings を
# 対象 PR 版で作り直してしまい、S-10 の意味が無くなる。
async def open_import_account_dialog(page, link, connect_btn, attempts=5):
    """［プロフィールからアカウントをインポート］を押し、［接続］が出るまで粘る。

    このリンクの click は knockout の `click: importAuth` バインド。アドオンを
    有効にした直後は mako がサーバ側で描画した HTML だけが先に見えていて
    applyBindings がまだ走っていないことがあり、その状態で押すとクリックは
    捨てられる。バインドの有無を覗くより押し直すほうが確実。
    """
    await expect(link).to_be_visible(timeout=transition_timeout)
    for i in range(attempts):
        await link.click(timeout=transition_timeout)
        try:
            await expect(connect_btn).to_be_enabled(timeout=10000)
            return
        except AssertionError:
            print(f'[i] ［接続］のダイアログが出ない({i + 1} 回目)。押し直す')
    raise AssertionError(
        '［プロフィールからアカウントをインポート］を'
        f' {attempts} 回押しても［接続］のダイアログが出ない')


async def confirm_import_account(page, connect_btn):
    """［接続］を押す。アカウントが複数あるときは何を選ぶのかを出す。"""
    select = page.locator('//select[contains(@class, "bootbox-input-select")]')
    if await select.count() > 0:
        print('[!] S3 アカウントが複数ある。既定(先頭)のまま接続する:',
              await select.locator('option').all_inner_texts())
    await asyncio.sleep(1)
    await connect_btn.click(timeout=transition_timeout)


async def no_addon_entry_point(page):
    """アドオン構成に入り口が無いときの AssertionError を作る。

    何が描かれていたのかが証跡だけで分かるように #<id>Scope のテキストを
    添える。例外は返すだけで、送出は呼び出し側に任せる。
    """
    scope = page.locator(f'//*[@id = "{addon_storage_id}Scope"]')
    if await scope.count() == 0:
        shown = f'#{addon_storage_id}Scope が描画されていない'
    else:
        text = (await scope.first.inner_text()).strip()
        if not text:
            text = ' '.join((await scope.first.text_content() or '').split())
        shown = text[:500] or '(テキストが無い)'
    return AssertionError(
        'アドオン構成に［アカウントに接続する］も［プロフィールから'
        'アカウントをインポート］も無く、バケットの選択も出ていない。'
        f'#{addon_storage_id}Scope の表示(先頭 500 文字): {shown}')


async def selected_bucket_of(page):
    """アドオン構成に表示されている選択中のバケット(folder_id)を読む。

    画面は `Current Bucket: <bucket>:/ (US Standard)` の形で出す。読めない
    ときは None を返す(呼び出し側が生のテキストを証跡に残す)。
    """
    scope = page.locator(f'//*[@id = "{addon_storage_id}Scope"]')
    await expect(scope).to_be_visible(timeout=transition_timeout)
    text = await scope.inner_text()
    found = re.search(r'(?:Current Bucket|現在のバケット)\s*[:：]\s*(\S+)', text)
    return found.group(1) if found else None


async def connect_s3_addon(page, project_name, bucket, reselect=True):
    """プロジェクトに S3 アドオンを接続し、ファイルページを開く。

    アカウント(アクセスキー)はユーザー単位に保持されるので、2 回目以降は
    「アカウントに接続する」ではなく既存アカウントの選択になる。両方に対応する。

    `reselect=False` は「既にアカウントが接続済み」の分岐でだけ効く。
    ［変更］→［保存］をやり直さず、表示されているバケットを読んで
    `entered_by` / `folder_id` / `reselected` を持つ dict を返す。
    選び直すと保存済みの `folder_id` を今の実装で書き直してしまい、
    「前の run が新規接続したときの保存形」が確かめられなくなるため。
    **ほかの分岐と `reselect=True`(既定)のときの挙動は変えていない**
    (戻り値も従来どおり入り口の種別の文字列)。
    """
    await open_project(page, project_name)

    await page.locator('//a[text() = "アドオン"]').click()
    container = page.locator(f'//div[@full_name = "{addon_storage_name}"]').first
    await expect(container).to_be_visible(timeout=transition_timeout)
    enable_link = page.locator(
        f'//div[@full_name = "{addon_storage_name}"]'
        '//descendant::a[text() = "有効にする"]')
    if await enable_link.count() == 0:
        print(f'[i] {addon_storage_name} は既に有効。有効化は飛ばす')
    else:
        # ［有効にする］の <a> は mako がサーバ側で描画しており、bootbox の
        # 確認ダイアログを出す click ハンドラは project-addons-page.js が
        # バンドル読み込み後に bind する。visible になった瞬間に押すと
        # クリックが捨てられるので、ハンドラが付くまで待つ。
        await page.wait_for_function(
            '(name) => {'
            ' const jq = window.jQuery; if (!jq) return false;'
            ' for (const b of document.getElementsByClassName("addon-container")) {'
            '   if (b.getAttribute("full_name") !== name) continue;'
            '   const a = b.querySelector("a"); if (!a) return false;'
            '   const ev = jq._data(a, "events");'
            '   return !!(ev && ev.click && ev.click.length);'
            ' }'
            ' return false; }',
            arg=addon_storage_name, timeout=transition_timeout)
        await enable_link.click()
        confirm = page.locator('//button[@data-bb-handler = "confirm"]')
        await expect(confirm).to_be_visible(timeout=transition_timeout)
        await confirm.click()
    await asyncio.sleep(2)

    input_creds = page.locator(f'#{addon_storage_id}InputCredentials')
    connect_link = page.locator(
        f'//img[@src = "/static/addons/{addon_storage_id}/comicon.png"]/..'
        f'//a[contains(text(), "アカウントに接続する")]')
    import_link = page.locator(
        f'//*[@id = "{addon_storage_id}Scope"]'
        '//a[contains(text(), "プロフィールからアカウントをインポート")]')
    # ［接続］は bootbox の確認ボタン。押し直しでダイアログが重なることが
    # あるので一番手前を見る。
    connect_btn = page.locator('//button[text() = "接続"]').last
    new_bucket = page.locator('//button[@id = "newBucket"]')

    # 「アドオンを構成」の中身は knockout が描く。［有効にする］の直後は
    # mako がサーバ側で描画した HTML だけが見えていたり、applyBindings 直後で
    # if: バインドの中身が消えたままだったりする(設定の取得が返るまで
    # showTokenCreateButton も showImport も false、#newBucket は
    # visible: バインドで隠れている)。この瞬間に count() / is_visible() を
    # 一度だけ見ると「どれも無い」と判定して落ちる。上の sleep は保険で、
    # 待ちの根拠にはしない。3 つのうちどれかが見えるまで待ってから分岐する。
    entry_point = connect_link.or_(import_link).or_(new_bucket).first
    try:
        await expect(entry_point).to_be_visible(timeout=transition_timeout)
    except AssertionError:
        raise await no_addon_entry_point(page)

    entered_by = None
    folder_id = None
    skip_select = False
    if await connect_link.count() > 0:
        entered_by = 'connect'
        await connect_link.first.click()
        await expect(input_creds.locator('.btn-success')).to_be_enabled(
            timeout=transition_timeout)
        await asyncio.sleep(1)
        await page.locator(
            f'//*[@id = "{addon_storage_id}InputCredentials"]'
            '//input[@id = "access_key"]').fill(s3_access_key)
        await page.locator(
            f'//*[@id = "{addon_storage_id}InputCredentials"]'
            '//input[@id = "secret_key"]').fill(s3_secret_key)
        await input_creds.locator('.btn-success').click()
        await expect(connect_btn).to_be_enabled(timeout=transition_timeout)
        await confirm_import_account(page, connect_btn)
    elif await import_link.count() > 0:
        entered_by = 'import'
        await open_import_account_dialog(page, import_link.first, connect_btn)
        await confirm_import_account(page, connect_btn)
    elif await new_bucket.is_visible():
        entered_by = 'already-connected'
        if reselect:
            print('[i] 既にアカウントが接続済み。バケット選択から続ける')
            await page.locator(
                '//button[.//span[text() = "変更" or text() = "Change"]]'
            ).click(timeout=transition_timeout)
        else:
            skip_select = True
            folder_id = await selected_bucket_of(page)
            print('[i] 既にアカウントが接続済み。選び直さない'
                  f'(表示されているバケット: {folder_id})')
    else:
        raise await no_addon_entry_point(page)

    if not skip_select:
        await expect(new_bucket).to_be_visible(timeout=transition_timeout)
        await page.locator(
            f'//span[text() = "{bucket}"]/../../..'
            '//input[@type = "radio"]').click()
        # delay を入れないと「正常にリンクされました」が出ない
        await asyncio.sleep(1)
        await page.locator(
            f'//*[@class = "{addon_storage_id}-confirm-selection"]'
            '//input[@value = "保存"]').click()
        await expect(page.locator(
            f'//*[@id = "{addon_storage_id}Scope"]'
            '/*[contains(@class, "help-block")]'
            f'//*[contains(text(), "正常にリンクされました")]')).to_be_visible(
                timeout=30000)

    try:
        await page.get_by_role('link', name='ファイルページ').click()
    except Exception:
        await page.locator('//*[@id = "projectNavFiles"]//a').click()
    await expect(page.locator('//*[@id = "treeGrid"]')).to_be_visible(
        timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(
        page, addon_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(2)
    print(f'{project_name}: {addon_storage_name} アドオンを'
          + ('接続済みのまま使う' if skip_select else '接続した')
          + f'(入り口: {entered_by})')
    # 呼び出し側が「既に接続済みだった」ことを証跡に書けるように返す。
    # 戻り値を使わない呼び出しはそのままで差し支えない。
    if skip_select:
        return {'entered_by': entered_by, 'folder_id': folder_id,
                'reselected': False}
    return entered_by

In [ ]:
# 回帰 NB セル [40] の複製。
def listing_pages(entries=None, path_contains=None):
    """WB 応答から「フォルダ一覧(?meta=)」だけを抜き、next_token の連鎖に直す。"""
    from urllib.parse import urlparse, parse_qs
    entries = wb_responses if entries is None else entries
    pages = []
    for e in entries:
        if e['method'] != 'GET' or 'meta=' not in e['url']:
            continue
        parsed = urlparse(e['url'])
        if path_contains and path_contains not in parsed.path:
            continue
        q = parse_qs(parsed.query)
        page_ = {
            'url_path': parsed.path,
            'request_next_token': q.get('next_token', [None])[0],
            'status': e['status'],
            'at': e['at'],
        }
        try:
            body = json.loads(e['body'])
        except Exception:
            page_['parse_error'] = True
            page_['names'] = []
            page_['response_next_token'] = None
        else:
            data = body.get('data', []) if isinstance(body, dict) else []
            page_['count'] = len(data)
            page_['names'] = [d.get('attributes', {}).get('name') for d in data]
            page_['response_next_token'] = (
                body.get('next_token') if isinstance(body, dict) else None)
        pages.append(page_)
    return pages


def list_versions(bucket, prefix, region=None):
    """回帰 NB セル [50] の複製。当該 Key の Versions / DeleteMarkers を全ページ集める。"""
    c = s3_client(region or s3_region_default)
    versions, markers = [], []
    kwargs = {'Bucket': bucket, 'Prefix': prefix}
    while True:
        r = c.list_object_versions(**kwargs)
        versions.extend({'Key': v['Key'], 'VersionId': v['VersionId'],
                         'IsLatest': v['IsLatest'], 'Size': v.get('Size'),
                         'ETag': v.get('ETag')}
                        for v in r.get('Versions', []))
        markers.extend({'Key': m['Key'], 'VersionId': m['VersionId'],
                        'IsLatest': m['IsLatest']}
                       for m in r.get('DeleteMarkers', []))
        if not r.get('IsTruncated'):
            break
        kwargs['KeyMarker'] = r.get('NextKeyMarker')
        kwargs['VersionIdMarker'] = r.get('NextVersionIdMarker')
    return {'prefix': prefix, 'Versions': versions, 'DeleteMarkers': markers,
            'version_count': len(versions), 'delete_marker_count': len(markers)}

In [ ]:
# メタデータの採取と突合(この NB 固有。複製元なし)。
#
# **一覧(ListObjects)と単体取得(HEAD)は WaterButler の中で別のクラスが作る。**
#   - 一覧      -> S3FileMetadata
#   - 単体取得  -> S3FileMetadataHeaders
# 同じオブジェクトなのに片方だけ ETag に引用符が残る、という食い違いが起きうる
# ところなので、S-10 では**両方**を採って突き合わせる。
META_KEYS = ('etag', 'size', 'md5', 'version')


def _meta_of(attributes):
    """WB の attributes から比べたい値だけを抜く。

    `modified` は比較対象にしない(読むたびに変わるわけではないが、
    develop 版と対象 PR 版で書式が違っても欠陥ではない)。
    """
    extra = attributes.get('extra') or {}
    hashes = extra.get('hashes') or {}
    return {
        'name': attributes.get('name'),
        'path': attributes.get('path'),
        'materialized': attributes.get('materialized'),
        'kind': attributes.get('kind'),
        'size': attributes.get('size'),
        'etag': attributes.get('etag'),
        'md5': extra.get('md5'),
        'hashes_md5': hashes.get('md5'),
        'version': extra.get('version'),
        'modified': attributes.get('modified'),
    }


def _unquote(value):
    return value.replace('"', '') if isinstance(value, str) else value


def _md5_of(meta):
    """比べるべき md5 を 1 つ返す。無ければ None。

    S3 アドオン(S3FileMetadata / S3FileMetadataHeaders)は `extra.md5` と
    `extra.hashes.md5` の両方に同じ値を入れるが、**機関ストレージ
    (osfstorage)は `extra.hashes.md5` にしか入れず `extra.md5` は
    None** になる。`md5` だけを見ると osfstorage の全件が「期待値と違う」に
    なってしまうので、None のときは `hashes_md5` を見る。
    """
    meta = meta or {}
    for field in ('md5', 'hashes_md5'):
        value = _unquote(meta.get(field))
        if value is not None:
            return value
    return None


def has_quotes(value):
    return isinstance(value, str) and '"' in value


async def wb_provider_root(page, provider):
    """今開いているプロジェクトの、その WB プロバイダのルート URL を作る。

    `https://<wb>/v1/resources/<guid>/providers/<provider>/` を組み立てる。

    **応答レコーダ(wb_responses / wb_all)には頼らない。** レコーダは
    page.on('response') なのでブラウザが出した要求しか拾わず、
    page.request.get の分は入らない。ファイルツリーの一覧も osf.io 側の
    grid API を通るので、プロジェクトを開いても WB の応答が 1 件も
    出ないことがある。空のまま wb_all に落ちると **前に開いた別の
    プロジェクト**のルートを掴む(run-20261016-205046 の事象)。
    osf.io は全ページの contextVars に WB の URL とノードの GUID を
    出しているので、そこから読めば取り違えない。
    """
    ctx = await page.evaluate(
        '() => { const c = window.contextVars || {};'
        ' return {wb: c.waterbutlerURL || null,'
        '         guid: (c.node && c.node.id) || null}; }')
    wb, guid = ctx.get('wb'), ctx.get('guid')
    assert wb and guid, (
        'contextVars から WB の URL とプロジェクトの GUID が読めない。'
        'プロジェクトのページを開いてから呼ぶこと。'
        f' 現在地: {page.url} / 読めた値: {ctx}')
    return f'{wb.rstrip("/")}/v1/resources/{guid}/providers/{provider}/'


async def wb_get_json(page, url):
    """page の資格情報で WB を直接叩いて JSON を取る。

    page.request はブラウザのコンテキスト(= Cookie)を共有するので、
    画面を操作せずに `?meta=` を取れる。
    """
    resp = await page.request.get(url, timeout=transition_timeout * 3)
    body = await resp.text()
    try:
        return resp.status, json.loads(body)
    except Exception:
        return resp.status, {'_raw': body[:2000]}


def meta_url_of(link):
    """WB のファイル URL に `?meta=` を付ける。"""
    return link + ('&' if '?' in link else '?') + 'meta='


async def fetch_listing(page, folder_url):
    """フォルダの一覧(?meta=)を取り、{name: (attributes, links)} にする。

    next_token が返る間は辿る(1000 件超のフォルダでも落ちないように)。
    """
    items = {}
    url = meta_url_of(folder_url)
    pages = 0
    while True:
        status, body = await wb_get_json(page, url)
        pages += 1
        assert status == 200, f'一覧が {status}: {url}'
        for d in body.get('data', []):
            attrs = d.get('attributes', {})
            items[attrs.get('materialized') or attrs.get('name')] = (
                attrs, d.get('links', {}))
        token = body.get('next_token')
        if not token:
            break
        base = meta_url_of(folder_url)
        url = f'{base}&next_token={token}'
    return items, pages


async def fetch_listing_by_name(page, folder_url):
    """フォルダの一覧(?meta=)を {名前: (attributes, links)} で返す。"""
    items, _ = await fetch_listing(page, folder_url)
    return {attrs.get('name'): (attrs, links)
            for attrs, links in items.values()}


async def resolve_folder(page, root, folders, cache):
    """ルートから folders を 1 段ずつ辿り、(そのフォルダの URL, 一覧) を返す。

    **osfstorage はパスがファイル ID なので、名前から URL を組み立てられない。**
    一覧に出てきた子の link を辿る形にすれば s3 でも osfstorage でも同じに
    なる。cache は一覧の取り直しを避けるための {たどった道: 一覧}。
    """
    url, key = root, ''
    if key not in cache:
        cache[key] = await fetch_listing_by_name(page, url)
    listing = cache[key]
    for name in folders:
        attrs, links = listing.get(name, (None, None))
        assert attrs is not None, (
            f'フォルダ "{name}" が一覧に無い({url})。'
            f' 一覧にあるもの: {sorted(listing)}')
        assert attrs.get('kind') == 'folder', f'"{name}" はフォルダではない'
        sub = links.get('upload') or links.get('new_folder')
        assert sub, f'フォルダ "{name}" の子の URL が一覧に無い'
        url = sub.split('?')[0]
        key = f'{key}/{name}'
        if key not in cache:
            cache[key] = await fetch_listing_by_name(page, url)
        listing = cache[key]
    return url, listing


async def resolve_file(page, root, rel, cache):
    """控えの相対パスから (attributes, links, 一覧の URL) を引く。

    見つからないときは (None, None, 理由) を返す。例外にしないのは、
    1 件の欠落で残りの採取を止めないため。
    """
    segments = [s for s in rel.split('/') if s]
    *folders, name = segments
    try:
        folder_url, listing = await resolve_folder(page, root, folders, cache)
    except AssertionError as e:
        return None, None, str(e)
    attrs, links = listing.get(name, (None, None))
    if attrs is None:
        return None, None, (f'一覧に無い: "{name}"'
                            f'({meta_url_of(folder_url)})。'
                            f' 一覧にあるもの: {sorted(listing)}')
    return attrs, links, meta_url_of(folder_url)


def scope_rel_paths(scope, files=None):
    """控え state['files'] から、その段に置いたファイルの相対パスを取る。

    控えのキーは `<段>/<相対パス>`(例 `addon/S-10-folder/サブ/nested.txt`)。
    **「一覧に見えたもの」ではなく「置いたもの」を起点にする**ので、
    採り漏らしが黙って消えない。
    """
    files = state.get('files', {}) if files is None else files
    return [key.split('/', 1)[1] for key, info in sorted(files.items())
            if info.get('where') == scope and '/' in key]


async def collect_metadata(page, folder_url, recurse=True, _depth=0):
    """フォルダ以下の全ファイルの「一覧側」と「単体取得側」のメタデータを集める。

    戻り値: {materialized_path: {'listing': {...}, 'headers': {...}}}
    """
    out = {}
    items, _ = await fetch_listing(page, folder_url)
    for key, (attrs, links) in sorted(items.items()):
        entry = {'listing': _meta_of(attrs)}
        if attrs.get('kind') == 'folder':
            if recurse and _depth < 5:
                sub = links.get('upload') or links.get('new_folder')
                if sub:
                    # upload リンクは `...?kind=file` なのでクエリを落とす
                    sub = sub.split('?')[0]
                    out.update(await collect_metadata(page, sub, recurse,
                                                      _depth + 1))
            out[key] = entry
            continue
        # 単体取得。ここだけが S3FileMetadataHeaders を通る。
        link = links.get('move') or links.get('download')
        if link:
            status, body = await wb_get_json(page, meta_url_of(link.split('?')[0]))
            entry['headers_status'] = status
            entry['headers'] = _meta_of(
                (body.get('data') or {}).get('attributes', {}))
        out[key] = entry
    return out


async def collect_metadata_paths(page, root, rel_paths):
    """控えのパスぶんだけ、一覧側と単体取得側のメタデータを採る。

    戻り値: {'/<相対パス>': {'listing': ..., 'headers': ...}}。
    引けなかったものは {'error': 理由} を残す(黙って欠かさない)。
    """
    out, cache = {}, {}
    for rel in rel_paths:
        attrs, links, where = await resolve_file(page, root, rel, cache)
        key = '/' + rel
        if attrs is None:
            out[key] = {'error': where}
            continue
        entry = {'listing': _meta_of(attrs), 'listing_url': where}
        # 単体取得。ここだけが S3FileMetadataHeaders を通る。
        link = links.get('move') or links.get('download')
        if link:
            url = meta_url_of(link.split('?')[0])
            status, body = await wb_get_json(page, url)
            entry['headers_url'] = url
            entry['headers_status'] = status
            entry['headers'] = _meta_of(
                (body.get('data') or {}).get('attributes', {}))
        out[key] = entry
    return out


def _s3_expected_of(s3_before, key):
    """metadata_a_s3 の 1 件を md5 / size の期待値に直す。

    develop 版の WB は S3 の ETag をそのまま md5 として返す
    (waterbutler/providers/s3/metadata.py)ので、引用符を外した ETag が
    md5 の期待値になる。
    """
    e = (s3_before or {}).get(key) or {}
    if not e.get('etag_s3'):
        return None
    return {'md5': e['etag_s3'], 'size': e.get('size'), 's3_key': e.get('key')}


def metadata_basis(before, after, s3_before=None):
    """キーごとに、何を md5 の期待値にしたかを返す。

    - `phase-a`: phase a で WB から実測した値(metadata_a)
    - `s3-etag`: phase a で boto3 から採った S3 の ETag(metadata_a_s3)
    - `none`: どちらも無い(比べられない)
    """
    out = {}
    for key in sorted(set(before) | set(after) | set(s3_before or {})):
        b = before.get(key) or {}
        if b.get('listing') or b.get('headers'):
            out[key] = 'phase-a'
        elif _s3_expected_of(s3_before, key):
            out[key] = 's3-etag'
        else:
            out[key] = 'none'
    return out


def _quoted_and_cross_checks(key, a, basis):
    """引用符付き ETag と、一覧側 vs 単体取得側の食い違い。期待値に依らない。"""
    out = []
    for side in ('listing', 'headers'):
        as_ = a.get(side) or {}
        if as_ and (has_quotes(as_.get('etag')) or has_quotes(as_.get('md5'))):
            out.append({'key': key, 'side': side, 'basis': basis,
                        'problem': 'quoted-etag',
                        'etag': as_.get('etag'), 'md5': as_.get('md5')})
    la, ha = (a.get('listing') or {}), (a.get('headers') or {})
    if ha and la.get('kind') != 'folder':
        if _unquote(la.get('etag')) != _unquote(ha.get('etag')):
            out.append({'key': key, 'side': 'listing-vs-headers',
                        'basis': basis, 'field': 'etag',
                        'listing': la.get('etag'),
                        'headers': ha.get('etag')})
        # md5 も `md5` が None なら `hashes_md5` で比べる(osfstorage)。
        if _md5_of(la) != _md5_of(ha):
            out.append({'key': key, 'side': 'listing-vs-headers',
                        'basis': basis, 'field': 'md5',
                        'listing': _md5_of(la),
                        'headers': _md5_of(ha)})
    return out


def metadata_diff(before, after, s3_before=None):
    """phase a と phase b のメタデータを突き合わせる。

    - 期待値は phase a の実測(`before`)が本筋。`before` に無いキーは
      phase a で S3 から直接採った `s3_before`(metadata_a_s3)の ETag を
      md5 の期待値に使う。**どちらを使ったかは各行の `basis` に書く。**
    - `etag` / `md5` / `size` の不一致を出す(`etag` は引用符を落としてから比べる)
    - md5 は `extra.md5` が None なら `extra.hashes.md5` で比べる。機関
      ストレージ(osfstorage)は後者にしか md5 を入れない
    - **引用符が残っていること自体を差分として出す**(落としてから比べると
      隠れてしまうので、別の欄に記録する)
    - 採れなかったもの(`error`)も差分として出す。黙って一致にしない
    """
    diffs = []
    basis = metadata_basis(before, after, s3_before)
    for key in sorted(set(before) | set(after) | set(s3_before or {})):
        b, a = before.get(key), after.get(key)
        if (b or {}).get('error') or (a or {}).get('error'):
            diffs.append({'key': key, 'basis': basis[key],
                          'problem': 'fetch-error',
                          'before': (b or {}).get('error'),
                          'after': (a or {}).get('error')})
            continue
        if a is None:
            diffs.append({'key': key, 'basis': basis[key],
                          'problem': 'missing',
                          'in_before': b is not None, 'in_after': False})
            continue
        if basis[key] == 'phase-a':
            for side in ('listing', 'headers'):
                bs, as_ = (b or {}).get(side) or {}, a.get(side) or {}
                if not bs and not as_:
                    continue
                for field in ('etag', 'size'):
                    bv, av = bs.get(field), as_.get(field)
                    if _unquote(bv) != _unquote(av):
                        diffs.append({'key': key, 'side': side, 'field': field,
                                      'basis': 'phase-a',
                                      'before': bv, 'after': av})
                # md5 は md5 / hashes_md5 のどちらか出ているほうで比べる。
                if _md5_of(bs) != _md5_of(as_):
                    diffs.append({'key': key, 'side': side, 'field': 'md5',
                                  'basis': 'phase-a',
                                  'before': _md5_of(bs), 'after': _md5_of(as_),
                                  'before_raw': {'md5': bs.get('md5'),
                                                 'hashes_md5': bs.get('hashes_md5')},
                                  'after_raw': {'md5': as_.get('md5'),
                                                'hashes_md5': as_.get('hashes_md5')}})
        elif basis[key] == 's3-etag':
            exp = _s3_expected_of(s3_before, key)
            for side in ('listing', 'headers'):
                as_ = a.get(side) or {}
                if not as_:
                    continue
                if _md5_of(as_) != exp['md5']:
                    diffs.append({'key': key, 'side': side, 'field': 'md5',
                                  'basis': 's3-etag',
                                  'expected': exp['md5'],
                                  'after': _md5_of(as_),
                                  'after_raw': {'md5': as_.get('md5'),
                                                'hashes_md5': as_.get('hashes_md5')}})
                if exp['size'] is not None and as_.get('size') is not None:
                    if int(as_['size']) != int(exp['size']):
                        diffs.append({'key': key, 'side': side, 'field': 'size',
                                      'basis': 's3-etag',
                                      'expected': exp['size'],
                                      'after': as_.get('size')})
        else:
            diffs.append({'key': key, 'basis': 'none',
                          'problem': 'no-expected-value'})
            continue
        diffs.extend(_quoted_and_cross_checks(key, a, basis[key]))
    return diffs


async def download_sha256(page, link):
    """WB からファイルを取り、SHA-256 と大きさを返す。

    `page.request.get` はリダイレクトを追うので、302 → presigned でも中身が
    取れる。302 そのものの確認は UI のツールバーからのダウンロード(後述)で
    別に採る。
    """
    url = link.split('?')[0]
    resp = await page.request.get(url, timeout=transition_timeout * 20)
    body = await resp.body()
    return {'status': resp.status, 'size': len(body),
            'sha256': sha256_bytes(body)}


async def collect_downloads(page, folder_url, recurse=True, _depth=0):
    """フォルダ以下の全ファイルを取り、SHA-256 を集める。"""
    out = {}
    items, _ = await fetch_listing(page, folder_url)
    for key, (attrs, links) in sorted(items.items()):
        if attrs.get('kind') == 'folder':
            if recurse and _depth < 5:
                sub = links.get('upload') or links.get('new_folder')
                if sub:
                    out.update(await collect_downloads(page, sub.split('?')[0],
                                                       recurse, _depth + 1))
            continue
        link = links.get('download') or links.get('move')
        if link:
            out[key] = await download_sha256(page, link)
    return out


async def collect_downloads_paths(page, root, rel_paths):
    """控えのパスぶんだけダウンロードして SHA-256 を採る。"""
    out, cache = {}, {}
    for rel in rel_paths:
        attrs, links, where = await resolve_file(page, root, rel, cache)
        key = '/' + rel
        if attrs is None:
            out[key] = {'error': where}
            continue
        link = links.get('download') or links.get('move')
        if not link:
            out[key] = {'error': 'ダウンロードの URL が一覧に無い'}
            continue
        print(f'    DL {key} ({attrs.get("size")} bytes)')
        out[key] = await download_sha256(page, link)
    return out

In [ ]:
# フォルダ作成。`取りまとめ-S3共通.ipynb` には fangorn の［新規フォルダ作成］の
# 手順が無かったので、`テスト手順-ストレージ共通-ファイル基本操作.ipynb` の
# セル [43] / [45] / [47] と同じ組み立てで新しく書いた。
async def create_folder(page, name):
    """今開いている行(ストレージ or フォルダ)の下に新規フォルダを作る。

    ［新規フォルダ作成］は選択中の行に対して出るので、呼ぶ前に親の行を
    クリックしておくこと。
    """
    new_folder_btn = page.locator(
        '//i[contains(@class, "fa-plus")]/../*[text() = "新規フォルダ作成"]')
    await expect(new_folder_btn).to_be_enabled(timeout=transition_timeout)
    await new_folder_btn.click()
    folder_input = page.locator('#createFolderInput')
    await expect(folder_input).to_be_editable(timeout=transition_timeout)
    await folder_input.fill(name)
    await folder_input.press('Enter')
    title = grdm.get_select_folder_title_locator(page, name)
    await title.scroll_into_view_if_needed()
    await expect(title).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)
    print(f'フォルダを作成した: {name}')


# 回帰 NB セル [31] の複製(302 → presigned の採取)。
import re

g10_redirects = []


def redact_presigned(url):
    """presigned URL の署名と資格情報を伏せる。

    SigV4 は `X-Amz-Signature` / `X-Amz-Credential` に、SigV2(boto 版)は
    `Signature` と `AWSAccessKeyId` に乗る。後者はクエリの区切りの直後だけを
    対象にして、`X-Amz-Signature` を二重に伏せないようにする。
    """
    out = re.sub(r'(X-Amz-(?:Signature|Credential|Security-Token)=)[^&\s]+',
                 r'\1***', url)
    return re.sub(r'([?&](?:Signature|AWSAccessKeyId)=)[^&\s]+',
                  r'\1***', out)


def redirect_entry(url, status, loc):
    return {
        'url': url,
        'status': status,
        # `direct` 付きの要求は WB が自分でストリームする仕様なので 302 に
        # ならない。「経路を踏んでいない」と「経路は踏んだが 302 にならない」を
        # 取り違えないために必ず残す。
        'has_direct': bool(re.search(r'[?&]direct(=|&|$)', url)),
        'location_host': loc.split('/')[2] if '://' in loc else '',
        'location_is_amazonaws': 'amazonaws.com' in loc,
        'location_has_signature': 'X-Amz-Signature' in loc,
        'location_has_sigv4_algorithm': 'AWS4-HMAC-SHA256' in loc,
        'location': redact_presigned(loc),
    }


async def capture_wb_redirects(page, out):
    """WB の 3xx を `Location` ごと拾う。

    応答レコーダ(上のセル)はヘッダを残さないので Location が取れない。
    HAR もダウンロードのリダイレクト経路を落とすことがあるため、302 の
    主信号はこれにする。
    """
    async def _on_response(response):
        if '/v1/resources/' not in response.url:
            return
        if response.status not in (301, 302, 303, 307, 308):
            return
        try:
            loc = (await response.all_headers()).get('location', '')
        except Exception as e:
            loc = f'<unavailable: {e!r}>'
        out.append(redirect_entry(response.url, response.status, loc))

    page.on('response', lambda r: asyncio.ensure_future(_on_response(r)))


def fangorn_download_button(page):
    """ファイル一覧のツールバーの［ダウンロード］。

    フォルダの「Zip 形式でダウンロード」も fa-download なので span の文字で絞る。
    """
    return page.locator(
        '//*[contains(@class, "fangorn-toolbar-icon")]'
        '[.//i[contains(@class, "fa-download")]]'
        '[.//span[text() = "ダウンロード"]]')


async def upload_overwrite_and_wait(page, path):
    """同名で上書きアップロードする(回帰 NB セル [51] の複製)。

    2 回目以降は fangorn が「置き換える / 保持する」を聞くので、出たら
    ［置き換える］を押す。押さないと仮行が残り、Dropzone は
    parallelUploads: 1 なので次のアップロードがキューで止まる。
    """
    name = os.path.basename(path)
    await clear_growls(page)
    await grdm.upload_file(page, path)
    await wait_for_upload_finished(page, name)
    keep = page.locator('//button[contains(text(), "置き換える")'
                        ' or contains(text(), "Replace")]')
    if await keep.count() > 0 and await keep.first.is_visible():
        await keep.first.click()
        await wait_for_upload_finished(page, name)
    await asyncio.sleep(2)
    return name


async def delete_file_from_toolbar(page, filename):
    """ファイル一覧のツールバーの［削除］で消す(回帰 NB セル [54] の複製)。"""
    await grdm.get_select_file_extension_locator(page, filename).click(
        timeout=transition_timeout)
    delete_btn = page.locator(
        '//i[contains(@class, "fa-trash")]/../*[text() = "削除"]')
    await expect(delete_btn).to_be_enabled(timeout=transition_timeout)
    await clear_growls(page)
    await delete_btn.click()
    confirm = page.locator(
        '//*[@id = "tb-tbody"]//*[@class = "modal-content"]'
        '//*[contains(@class, "btn-danger")]')
    await expect(confirm).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(1)
    await confirm.click()
    await expect(grdm.get_select_file_title_locator(page, filename)
                 ).to_have_count(0, timeout=transition_timeout * 3)
    print('growl:', await growl_messages(page))


async def download_from_toolbar(page, filename):
    """一覧のツールバーからダウンロードして保存先を返す。

    詳細画面の［ダウンロード］は OSF が `direct` を付けるので 302 にならない。
    presigned への 302 を採るには一覧のツールバーを使うこと。
    """
    download_dir = os.path.join(work_dir, 'downloaded')
    os.makedirs(download_dir, exist_ok=True)
    await grdm.get_select_file_extension_locator(page, filename).click(
        timeout=transition_timeout)
    btn = fangorn_download_button(page)
    await expect(btn).to_be_visible(timeout=transition_timeout)
    async with page.expect_download(timeout=transition_timeout * 10) as info:
        await btn.click(timeout=transition_timeout)
    download = await info.value
    path = os.path.join(download_dir, download.suggested_filename)
    await download.save_as(path)
    await asyncio.sleep(2)
    return path

## phase a: 仕込み(**試験環境は develop 版**)

旧実装で機関ストレージを登録し、S3 アドオンを接続し、データを置く。
置いたものの**メタデータとダウンロードの SHA-256 を控える**のがこの段の目的で、
突合は phase b で行う。

ここから `phase == 'a'` のセルだけが動く。phase b / c では全部スキップされる。

置くもの:

| 置き場所 | ファイル | 何を見たいか |
|---|---|---|
| 機関ストレージ(`*-inst`) | `S-10-ascii.txt`、`S-10-日本語.txt` | 機関ストレージ経由の読み書き |
| S3 アドオン(`*-addon`) | `S-10-ascii.txt`、`S-10-日本語 名前+記号(1).txt`、`S-10 space.txt`、`S-10-folder/サブ/nested.txt`、boto3 で直接置く `S-10-raw/a+b %25 c~d.txt`、200MB | 署名の対象になる文字、2 階層、GRDM 外から置かれたキー、マルチパートの ETag |
| S3 アドオン(`*-ver`、バージョニング有効) | `S-10-ver.txt` を 3 版 | バージョン一覧と削除 |
| (任意)S3 アドオン(`*-dot`) | `S-10-ascii.txt` | ドット入りバケット名 |

In [ ]:
if phase == 'a':
    async def _step(page):
        await admin_login(page)
        await goto_institutional_storage(page)
        await select_institution(page)
        before = await capture_institutional_storage_state(
            page, 'S-10a-institutional-storage-before')
        print('登録前:', before)
        state['institutional_before'] = before
        await register_institutional_s3(page, s3_bucket_mig, 'S-10a')
        await admin_logout(page)

    await run_pw(_step)
    save_state(state)
else:
    print(f'phase={phase} のためスキップ(機関ストレージは phase a でしか触らない)')

In [ ]:
# ユーザ画面へ。phase b / c でも使うので、ここは phase で分けない。
async def _step(page):
    await user_login(page)
    await capture_wb_redirects(page, g10_redirects)

await run_pw(_step)

In [ ]:
if phase == 'a':
    # S-10a: 機関ストレージのプロジェクトに 2 件置く。
    _inst_files = {}
    for _name, _body in (
            (f_ascii, f'S-10 institutional ascii\nrun_id: {run_id}\n'),
            (f_japanese, f'S-10 機関ストレージ 日本語\nrun_id: {run_id}\n')):
        _p = os.path.join(work_dir, _name)
        with open(_p, 'w') as _f:
            _f.write(_body)
        _inst_files[_name] = _p

    async def _step(page):
        await create_project_and_open_files(page, p_inst)
        wb_responses.clear()
        await open_institutional_storage(page)
        for _n, _p in _inst_files.items():
            await upload_and_wait(page, _p)

    await run_pw(_step)
    keep_all('S-10a-inst-upload.json')
    for _n, _p in _inst_files.items():
        state['files'][f'inst/{_n}'] = {
            'where': 'inst', 'name': _n,
            'sha256': sha256_file(_p), 'size': os.path.getsize(_p)}
    save_state(state)
    print(json.dumps(state['files'], indent=2, ensure_ascii=False))

In [ ]:
if phase == 'a':
    # S-10a: アドオンのプロジェクトを作り、develop 版で接続する。
    async def _step(page):
        await create_project_and_open_files(page, p_addon)
        await connect_s3_addon(page, p_addon, s3_bucket_mig)

    await run_pw(_step)
    keep_all('S-10a-addon-connect.json')

In [ ]:
if phase == 'a':
    # S-10a: 名前に意味のあるファイルを置く。
    # SigV2 と SigV4 でエンコードの規則が変わるので、空白・`+`・括弧・日本語を
    # 含む名前をわざと使う。
    _addon_files = {}
    for _name in (f_ascii, f_symbols, f_space):
        _p = os.path.join(work_dir, _name)
        with open(_p, 'w') as _f:
            _f.write(f'S-10 addon {_name}\nrun_id: {run_id}\n')
        _addon_files[_name] = _p

    _nested_path = os.path.join(work_dir, f_nested)
    with open(_nested_path, 'w') as _f:
        _f.write(f'S-10 nested\nrun_id: {run_id}\n')

    async def _step(page):
        await open_project_files(page, p_addon)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        for _n, _p in _addon_files.items():
            await upload_and_wait(page, _p)
        # 2 階層のフォルダを作り、底にファイルを置く
        await grdm.get_select_storage_title_locator(
            page, addon_storage_name).click(timeout=transition_timeout)
        await create_folder(page, folder_top)
        await grdm.get_select_folder_title_locator(
            page, folder_top).click(timeout=transition_timeout)
        await create_folder(page, folder_sub)
        await grdm.get_select_folder_title_locator(
            page, folder_sub).click(timeout=transition_timeout)
        await upload_and_wait(page, _nested_path)

    await run_pw(_step)
    keep_all('S-10a-addon-upload.json')

    for _n, _p in _addon_files.items():
        state['files'][f'addon/{_n}'] = {
            'where': 'addon', 'name': _n,
            'sha256': sha256_file(_p), 'size': os.path.getsize(_p)}
    state['files'][f'addon/{folder_top}/{folder_sub}/{f_nested}'] = {
        'where': 'addon', 'name': f_nested,
        'folder': f'{folder_top}/{folder_sub}',
        'sha256': sha256_file(_nested_path),
        'size': os.path.getsize(_nested_path)}
    save_state(state)

In [ ]:
if phase == 'a':
    # S-10a: GRDM の外から置かれたキーの代表。`+`・空白・`%25`・`~` を含む。
    # GRDM の画面からは作れない名前なので boto3 で直接置く。
    _raw_body = f'S-10 raw key\nrun_id: {run_id}\n'.encode()
    s3_client(s3_region_default).put_object(
        Bucket=s3_bucket_mig, Key=f_raw_key, Body=_raw_body)
    state['files'][f'addon/{f_raw_key}'] = {
        'where': 'addon', 'name': os.path.basename(f_raw_key),
        'key': f_raw_key, 'put_by': 'boto3',
        'sha256': sha256_bytes(_raw_body), 'size': len(_raw_body)}
    save_state(state)
    print(f'boto3 で直接置いた: {f_raw_key} ({len(_raw_body)} bytes)')

In [ ]:
if phase == 'a' and run_chunked:
    # S-10a: 200MB。develop 版(boto)のマルチパートで上がるので ETag が `-N` になる。
    # N は phase b で一致を見るので控える。
    assert chunked_file_size >= wb_contiguous_upload_size_limit, (
        f'chunked_file_size={chunked_file_size} は '
        f'CONTIGUOUS_UPLOAD_SIZE_LIMIT={wb_contiguous_upload_size_limit} 未満なので'
        '単発 PUT になり、マルチパートの ETag にならない')
    _chunk_path = os.path.join(work_dir, f_chunked)
    with open(_chunk_path, 'wb') as _f:
        _remain = chunked_file_size
        while _remain > 0:
            _n = min(_remain, 4 * 1024 * 1024)
            _f.write(os.urandom(_n))
            _remain -= _n
    print(_chunk_path, os.path.getsize(_chunk_path), 'bytes')

    async def _step(page):
        await open_project_files(page, p_addon)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        await upload_and_wait(page, _chunk_path)

    await run_pw(_step)
    keep_all('S-10a-chunked-upload.json')

    _h = s3_client(s3_region_default).head_object(
        Bucket=s3_bucket_mig, Key=f_chunked)
    _etag = _h['ETag'].strip('"')
    state['files'][f'addon/{f_chunked}'] = {
        'where': 'addon', 'name': f_chunked,
        'sha256': sha256_file(_chunk_path), 'size': os.path.getsize(_chunk_path)}
    state['expected']['chunked'] = {
        'etag_raw': _h['ETag'], 'etag': _etag,
        'multipart': '-' in _etag,
        'part_count': int(_etag.split('-')[1]) if '-' in _etag else None,
        'ContentLength': _h['ContentLength'],
    }
    save_state(state)
    record_json('S-10a-chunked.json', state['expected']['chunked'], quiet=False)
    assert state['expected']['chunked']['multipart'], (
        f'ETag にパート数の接尾辞が無い({_etag})。単発 PUT で上がっている。'
        'chunked_file_size を試験環境の CONTIGUOUS_UPLOAD_SIZE_LIMIT 以上にすること')
elif phase == 'a':
    print('run_chunked=False のためスキップ')

In [ ]:
if phase == 'a':
    # S-10a: バージョニング有効バケットに 3 版作る。
    _ver_path = os.path.join(work_dir, f_ver)

    async def _step(page):
        await create_project_and_open_files(page, p_ver)
        await connect_s3_addon(page, p_ver, s3_bucket_migver)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        for _i in (1, 2, 3):
            with open(_ver_path, 'w') as _f:
                _f.write(f'S-10 version {_i}\nrun_id: {run_id}\n')
                _f.write('x' * (100 * _i))
            await upload_overwrite_and_wait(page, _ver_path)
            print(f'  {_i} 回目のアップロード完了')
        await expect(grdm.get_select_file_title_locator(page, f_ver)
                     ).to_be_visible(timeout=transition_timeout)

    await run_pw(_step)
    keep_all('S-10a-ver-upload.json')

    _v = list_versions(s3_bucket_migver, f_ver)
    record_json('S-10a-versions.json', _v, quiet=False)
    state['files'][f'ver/{f_ver}'] = {
        'where': 'ver', 'name': f_ver,
        'sha256': sha256_file(_ver_path), 'size': os.path.getsize(_ver_path),
        'version_count': _v['version_count']}
    save_state(state)
    assert _v['version_count'] >= 2, (
        f'{f_ver} の版が {_v["version_count"]} 件しかない。'
        'バージョニングが効いていないか、上書きが 1 回も成立していない')

In [ ]:
if phase == 'a' and run_dotted:
    # S-10a(任意): ドット入りバケット名。**失敗しても止めない。**
    # develop 版でも通らないなら、phase b の S-10b-10 は比較対象なしとして扱う。
    _dot_path = os.path.join(work_dir, f_ascii)
    _dot_result = {'bucket': s3_bucket_dot, 'project': p_dot}
    try:
        async def _step(page):
            await create_project_and_open_files(page, p_dot)
            await connect_s3_addon(page, p_dot, s3_bucket_dot)
            wb_responses.clear()
            await open_storage(page, addon_storage_name)
            await upload_and_wait(page, _dot_path)

        await run_pw(_step)
        _dot_result['ok'] = True
    except Exception as _e:
        _dot_result['ok'] = False
        _dot_result['error'] = repr(_e)
        print(f'[!] ドット入りバケットの接続/アップロードが失敗した: {_e!r}')
        print('    記録して続行する(S-10b-10 は比較対象なしとして扱う)')
    _dot_result['wb_failures'] = [
        {'status': e['status'], 'url': e['url']} for e in wb_failures()]
    keep_all('S-10a-dot-wb.json')
    record_json('S-10a-dot.json', _dot_result, quiet=False)
    state['dot'] = _dot_result
    save_state(state)
elif phase == 'a':
    print('run_dotted=False のためスキップ')

In [ ]:
if phase == 'a':
    # S-10a: 置いたもの全部のメタデータとダウンロード SHA-256 を控える。
    # phase b はここで採った値と突き合わせる。
    a_meta = {}
    a_down = {}

    async def _step(page):
        global a_meta, a_down
        for _scope, _project, _provider, _opener in (
                ('inst', p_inst, 'osfstorage', open_institutional_storage),
                ('addon', p_addon, addon_storage_id, None),
                ('ver', p_ver, addon_storage_id, None)):
            await open_project_files(page, _project)
            # wb_responses を空にするのは証跡を段ごとに分けるためだけ。
            # ルートの決定には使わない(contextVars から採る)。
            wb_responses.clear()
            if _opener is None:
                await open_storage(page, addon_storage_name)
            else:
                await _opener(page)
            _root = await wb_provider_root(page, _provider)
            _paths = scope_rel_paths(_scope)
            print(f'{_scope}: {_root}')
            print(f'  控えにあるパス {len(_paths)} 件: {_paths}')
            a_meta[_scope] = await collect_metadata_paths(page, _root, _paths)
            a_down[_scope] = await collect_downloads_paths(page, _root, _paths)

    await run_pw(_step)
    keep_all('S-10a-metadata-fetch.json')
    record_json('S-10a-metadata.json', a_meta)
    record_json('S-10a-downloads.json', a_down)
    state['metadata_a'] = a_meta
    state['downloads_a'] = a_down
    save_state(state)
    print('採ったキー:')
    for _scope, _items in a_meta.items():
        print(f'  {_scope}: {sorted(_items)}')

In [ ]:
if phase == 'a':
    # S-10a の判定。ここは「仕込みが成立したか」だけを見る。
    _bad = wb_failures(wb_all)
    _verdict = [
        f'phase: a (run_id {run_id})',
        f'機関ストレージ: {s3_bucket_mig}',
        f'アドオン(通常): {s3_bucket_mig} / プロジェクト {p_addon}',
        f'アドオン(バージョン): {s3_bucket_migver} / プロジェクト {p_ver}',
        f'ドット入り: {s3_bucket_dot}',
        f'置いたファイル: {len(state["files"])} 件',
        f'WB 応答の 4xx/5xx・署名系の文言: {len(_bad)} 件',
    ]
    for _e in _bad:
        _verdict.append(f'  - {_e["status"]} {_e["method"]} {_e["url"]}')
    record('S-10a-verdict.txt', '\n'.join(_verdict))
    record_json('S-10a-wb-all.json', json.loads(_redact(
        json.dumps(wb_all, ensure_ascii=False))))
    assert not _bad, 'phase a の時点で WB 応答に失敗がある。仕込みが成立していない'
    print()
    print('=' * 74)
    print(' phase a 完了。次にすること')
    print('=' * 74)
    print(' 1. 試験環境を **対象 PR 版** に切り替える')
    print(' 2. このノートブックを phase="b" で実行する。そのとき')
    print(f'    state_path="{state_path}" を必ず渡すこと')
    print(' 3. バケットもプロジェクトも消さないこと')
    print('=' * 74)

In [ ]:
# phase a の基準値(S3 側の実測)。**phase を見ないので単独で実行できる。**
# phase a を流した run のカーネルに、後からこのセルだけを足して実行してよい。
#
# なぜ要るか: WB 経由のメタデータ採取が取りこぼしたキーでも、S3 を直接
# 見れば md5 の期待値が作れる。develop 版の WaterButler は S3 の ETag を
# そのまま md5 として返す(waterbutler/providers/s3/metadata.py の
# S3FileMetadata.extra / S3FileMetadataHeaders.extra がどちらも
# `md5 = self.raw['ETag'].replace('"', '')`)。つまり**引用符を外した
# ETag がそのまま md5 の期待値**になる。200MB のようなマルチパートの
# オブジェクトは `-<パート数>` が付く。
#
# 応答 JSON の `etag` のほうは WB が sha256('<provider>::<etag>') に
# 作り直した値(waterbutler/core/metadata.py)なので、S3 の ETag とは
# 直接比べられない。比べるのは extra.md5 / extra.hashes.md5。
_s3_now = {}
_s3_buckets = state.get('buckets', {})
_s3_region = _s3_buckets.get('region') or s3_region_default
# 段ごとのバケット。phase a の仕込みと同じ割り当て。
_scope_bucket = {
    'inst': _s3_buckets.get('mig'),
    'addon': _s3_buckets.get('mig'),
    'ver': _s3_buckets.get('migver'),
}


def _s3_key_of(scope, rel, info):
    """段ごとの S3 のキー。

    - S3 アドオン(addon / ver)はバケット直下にファイルのパスがそのまま
      入る。`S-10-raw/a+b %25 c~d.txt` のような生のキーもそのまま。
    - 機関ストレージ(inst)は osfstorage 経由なので、オブジェクト名は
      **中身の SHA-256** になる(waterbutler/providers/osfstorage/provider.py
      の `complete_name = stream.writers['sha256'].hexdigest`)。控えに
      記録してある sha256 がそのままキー。
    """
    return info.get('sha256') if scope == 'inst' else rel


for _key, _info in sorted(state.get('files', {}).items()):
    if '/' not in _key:
        continue
    _scope, _rel = _key.split('/', 1)
    _path = '/' + _rel
    _bucket = _scope_bucket.get(_scope)
    _s3key = _s3_key_of(_scope, _rel, _info)
    _entry = {'scope': _scope, 'bucket': _bucket, 'key': _s3key}
    if not _bucket or not _s3key:
        _entry['error'] = f'バケットかキーが控えから決まらない({_key})'
    else:
        try:
            # バージョン付きバケットでも VersionId を渡さなければ最新版。
            _h = s3_client(_s3_region).head_object(Bucket=_bucket, Key=_s3key)
        except Exception as _e:
            _entry['error'] = _redact(repr(_e))
        else:
            _entry['etag_raw'] = _h['ETag']
            _entry['etag_s3'] = _h['ETag'].replace('"', '')
            _entry['size'] = _h['ContentLength']
            _entry['multipart'] = '-' in _entry['etag_s3']
            _entry['version_id'] = _h.get('VersionId')
    _s3_now.setdefault(_scope, {})[_path] = _entry

record_json('S-10a-metadata-s3.json', _s3_now)
if state.get('metadata_a_s3') and phase != 'a':
    print('[!] 控えに metadata_a_s3 が既にある。phase a の基準値は上書きしない'
          '(採り直した値はこの run の証跡にだけ残す)')
else:
    state['metadata_a_s3'] = _s3_now
    save_state(state)
for _scope, _items in sorted(_s3_now.items()):
    print(f'{_scope}: バケット {_scope_bucket.get(_scope)}')
    for _p, _v in sorted(_items.items()):
        print(f'  {_p} -> {_v.get("etag_s3") or _v.get("error")}'
              f' / {_v.get("size")} bytes')

## phase b: 検証(**試験環境は対象 PR 版**)

phase a が develop 版で作った設定とデータを、**設定を一切触らずに**開く。

### 呼んではいけないもの

- `register_institutional_s3()` — 呼ぶと機関ストレージ設定が対象 PR 版で
  作り直され、「develop 版が作った設定を読めるか」が確かめられなくなる
- `connect_s3_addon()` — 同じく `NodeSettings` が作り直される。
  **S-10b-11 の新規プロジェクトに対してだけ**呼ぶ

判定は 1 項目ずつ結果を溜めてから最後にまとめて `assert` する。
**途中の 1 件で例外を投げて止めない**(残りの項目の結果が取れなくなるため)。

| 番号 | 見るもの |
|---|---|
| S-10b-1 | 機関ストレージ設定の表示 |
| S-10b-2 | `*-inst` の一覧・ダウンロード・302 |
| S-10b-3 | `*-addon` の一覧(フォルダ展開を含む) |
| S-10b-4 | `?meta=`(単体取得)と一覧の突合、**ETag の引用符** |
| S-10b-5 | 全ファイルのダウンロード SHA-256 |
| S-10b-6 | intra move → copy |
| S-10b-7 | バージョン付きファイルの UI 削除 |
| S-10b-8 | 新データ投入(phase c 用) |
| S-10b-9 | WB 応答に 4xx/5xx と署名系の文言が無いこと |
| S-10b-10(任意) | ドット入りバケット |
| S-10b-11 | 新規接続と `folder_id` の保存形 |

In [ ]:
# 判定の控え。各セルがここに結果を足し、最後のセルでまとめて判定する。
b_results = {}


def b_record(num, ok, detail):
    b_results[num] = {'ok': bool(ok), 'detail': detail}
    print(f'{num}: {"PASS" if ok else "FAIL"} — {detail}')
    return ok

In [ ]:
if phase == 'b':
    # S-10b-1: 機関ストレージ設定を**表示だけ**する。保存ボタンは押さない。
    #
    # `run_pw` に渡す関数は**値を返さないこと**。返した値は「新しいページ」と
    # みなされて Playwright のページ一覧に積まれる(scripts/playwright.py の
    # `next_page`)。結果は外の変数に入れる。
    _b1 = {}

    async def _step(page):
        await admin_login(page)
        await goto_institutional_storage(page)
        await select_institution(page)
        _b1['after'] = await capture_institutional_storage_state(
            page, 'S-10b-institutional-storage')
        await admin_logout(page)

    await run_pw(_step)
    _after = _b1.get('after') or {}
    _before = state.get('institutional_before') or {}
    _same_provider = (_after or {}).get('selected') == 's3'
    _diff = {k: {'phase_a': _before.get(k), 'phase_b': (_after or {}).get(k)}
             for k in set(_before) | set(_after or {})
             if _before.get(k) != (_after or {}).get(k)}
    record_json('S-10b-institutional-storage-diff.json',
                {'phase_a_before_register': _before, 'phase_b': _after,
                 'diff': _diff}, quiet=False)
    b_record('S-10b-1', _same_provider,
             f'選択は {(_after or {}).get("selected")} / 表示名 '
             f'{(_after or {}).get("storage_name")!r} / 差分 {len(_diff)} 件')

In [ ]:
if phase == 'b':
    # S-10b-2 / 3 / 4 / 5: 一覧・単体取得・ダウンロードを一度に採る。
    # phase a の控えと比べるのは次のセル。
    b_meta = {}
    b_down = {}
    b_listing = {}
    b_toolbar = {}

    async def _step(page):
        # 302 → presigned を踏ませる代表ファイルは段ごとに違う
        # (*-ver に S-10-ascii.txt は無い)。
        for _scope, _project, _provider, _opener, _rep in (
                ('inst', p_inst, 'osfstorage', open_institutional_storage,
                 f_ascii),
                ('addon', p_addon, addon_storage_id, None, f_ascii),
                ('ver', p_ver, addon_storage_id, None, f_ver)):
            await open_project_files(page, _project)
            wb_responses.clear()
            if _opener is None:
                await open_storage(page, addon_storage_name)
            else:
                await _opener(page)
            _root = await wb_provider_root(page, _provider)
            b_listing[_scope] = listing_pages()
            _paths = scope_rel_paths(_scope)
            print(f'{_scope}: {_root}')
            b_meta[_scope] = await collect_metadata_paths(page, _root, _paths)
            b_down[_scope] = await collect_downloads_paths(page, _root, _paths)
            # 302 → presigned は UI のツールバーからのダウンロードでしか出ない。
            # 代表 1 件だけ踏む。
            try:
                _path = await download_from_toolbar(page, _rep)
                b_toolbar[_scope] = {'file': _rep,
                                     'saved': os.path.basename(_path),
                                     'sha256': sha256_file(_path),
                                     'size': os.path.getsize(_path)}
            except Exception as _e:
                b_toolbar[_scope] = {'file': _rep, 'error': repr(_e)}

    await run_pw(_step)
    keep_all('S-10b-fetch.json')
    record_json('S-10b-listing.json', b_listing)
    record_json('S-10b-metadata.json', b_meta)
    record_json('S-10b-downloads.json', b_down)
    record_json('S-10b-toolbar-downloads.json', b_toolbar, quiet=False)

In [ ]:
if phase == 'b':
    # S-10b-3: phase a で置いたキーが欠落・文字化けなく出ているか。
    _a_meta = state.get('metadata_a', {})
    _a_s3 = state.get('metadata_a_s3', {})
    _a_down = state.get('downloads_a', {})
    _files = state.get('files', {})

    # 判定の対象から外すものを先に決める。どちらも「再実行だから合わない」
    # だけで、試験対象の退行ではない。
    # (1) b-7 で削除済みのキー: 版付きバケットに版が 1 つも無ければ、前の run の
    #     b-7 が消したということ。**消えているのが正しい**ので、欠落や
    #     「引けなかったもの」に数えると再実行のたびに必ず落ちる。
    # (2) phase b で置いたもの(placed_in_phase == 'b'): 控えの sha256 / size は
    #     **前の run の投入値**で、この run が b-8 で置き直す前の中身とは
    #     一致しない。新データの突合は b-8 が同じ run の値で行う。
    _excluded = {}

    def _exclude(scope, key, why):
        _excluded.setdefault(scope, {})[key] = why

    if list_versions(s3_bucket_migver, f_ver)['version_count'] == 0:
        _exclude('ver', f'/{f_ver}', 'b-7 で削除済み')
    for _fkey, _finfo in sorted(_files.items()):
        if _finfo.get('placed_in_phase') == 'b' and '/' in _fkey:
            _fscope, _frel = _fkey.split('/', 1)
            _exclude(_fscope, f'/{_frel}', 'phase b で投入(突合は b-8)')

    def _drop_excluded(per_scope):
        """scope ごとの dict から、判定対象外のキーを落とした写しを返す。"""
        return {_s: {_k: _v for _k, _v in (_items or {}).items()
                     if _k not in _excluded.get(_s, {})}
                for _s, _items in (per_scope or {}).items()}

    _a_meta = _drop_excluded(_a_meta)
    _a_s3 = _drop_excluded(_a_s3)
    _a_down = _drop_excluded(_a_down)
    _b_meta = _drop_excluded(b_meta)
    _b_down = _drop_excluded(b_down)
    _n_excluded = sum(len(_v) for _v in _excluded.values())

    _scopes = sorted(set(_a_meta) | set(_a_s3) | set(_b_meta))
    _basis = {_s: metadata_basis(_a_meta.get(_s, {}), _b_meta.get(_s, {}),
                                 _a_s3.get(_s, {}))
              for _s in _scopes}
    _missing = {}
    for _scope in _scopes:
        _now = set(_b_meta.get(_scope, {}))
        _missing[_scope] = sorted(set(_a_meta.get(_scope, {})) - _now)
    # 「phase a に無かった」は参考情報。**判定には使わない。** 期待値の基準が
    # s3-etag のキーは、phase a が WB では採れなかっただけで S3 からは採れて
    # いる(metadata_a_s3 に入っている)ので、ここには数えない。数えると
    # 控えを補正した run がまるごと「増えた」ことになってしまう。
    _extra = {}
    for _scope in _scopes:
        _extra[_scope] = sorted(
            k for k in set(_b_meta.get(_scope, {})) - set(_a_meta.get(_scope, {}))
            if _basis[_scope].get(k) != 's3-etag')
    record_json('S-10b-listing-diff.json',
                {'missing': _missing, 'extra': _extra,
                 'basis': _basis, 'excluded': _excluded}, quiet=False)
    # 控え駆動の採取なので、引けなかったものは error として残っている。
    _unreadable = {_scope: sorted(k for k, v in _items.items()
                                  if (v or {}).get('error'))
                   for _scope, _items in _b_meta.items()}
    b_record('S-10b-3',
             not any(_missing.values()) and not any(_unreadable.values()),
             f'欠落 {sum(len(v) for v in _missing.values())} 件 / '
             f'引けなかったもの {sum(len(v) for v in _unreadable.values())} 件'
             f'(参考: phase a に無かったもの '
             f'{sum(len(v) for v in _extra.values())} 件。判定には使わない / '
             f'対象外 {_n_excluded} 件)')

    # S-10b-4: メタデータの突合。引用符付き ETag はそれ自体を差分として出す。
    # 期待値は phase a の実測(metadata_a)が本筋。phase a が取りこぼした
    # キーは、同じ phase a で S3 から直接採った metadata_a_s3 の ETag を
    # md5 の期待値に使う。どちらを使ったかは basis に残す。
    _diffs = {}
    for _scope in _scopes:
        _items = _a_meta.get(_scope, {})
        _s3 = _a_s3.get(_scope, {})
        _diffs[_scope] = metadata_diff(_items, _b_meta.get(_scope, {}), _s3)
    record_json('S-10b-metadata-diff.json',
                {'diffs': _diffs, 'basis': _basis,
                 'unreadable': _unreadable, 'excluded': _excluded}, quiet=False)
    _quoted = [d for ds in _diffs.values() for d in ds
               if d.get('problem') == 'quoted-etag']
    _n_diff = sum(len(v) for v in _diffs.values())
    b_record('S-10b-4', _n_diff == 0,
             f'差分 {_n_diff} 件(うち引用符付き ETag {len(_quoted)} 件 / '
             f'対象外 {_n_excluded} 件)')

    # 200MB のパート数。
    if run_chunked and state.get('expected', {}).get('chunked'):
        _exp = state['expected']['chunked']
        _h = s3_client(s3_region_default).head_object(
            Bucket=s3_bucket_mig, Key=f_chunked)
        _now_etag = _h['ETag'].strip('"')
        record_json('S-10b-chunked.json',
                    {'phase_a': _exp, 'phase_b_etag_raw': _h['ETag'],
                     'phase_b_etag': _now_etag}, quiet=False)
        b_record('S-10b-4-multipart', _now_etag == _exp['etag'],
                 f'S3 側の ETag {_now_etag} / phase a {_exp["etag"]}')

    # S-10b-5: ダウンロードの SHA-256。
    # 期待値は phase a のダウンロード(downloads_a)が本筋。phase a が
    # 取りこぼしたキーは、控えに残した**投入時の SHA-256**を期待値にする。
    _mismatch = []
    for _scope in sorted(set(_a_down) | set(_b_down)):
        for _key in sorted(set(_a_down.get(_scope) or {})
                           | set(_b_down.get(_scope) or {})):
            _before = (_a_down.get(_scope) or {}).get(_key) or {}
            _now = (_b_down.get(_scope) or {}).get(_key)
            _want = _before.get('sha256') or (
                _files.get(f'{_scope}{_key}') or {}).get('sha256')
            if _now is None or _now.get('error'):
                _mismatch.append({'scope': _scope, 'key': _key,
                                  'problem': 'not-downloadable',
                                  'error': (_now or {}).get('error')})
            elif _want is None:
                _mismatch.append({'scope': _scope, 'key': _key,
                                  'problem': 'no-expected-value',
                                  'after': _now})
            elif _now.get('sha256') != _want:
                _mismatch.append({'scope': _scope, 'key': _key,
                                  'expected': _want, 'after': _now})
    record_json('S-10b-downloads-diff.json', _mismatch, quiet=False)
    b_record('S-10b-5', not _mismatch,
             f'不一致 {len(_mismatch)} 件(対象外 {_n_excluded} 件)')

    # S-10b-2: 機関ストレージの一覧とダウンロード、302 → presigned。
    _inst_ok = (not _missing.get('inst') and
                not [m for m in _mismatch if m['scope'] == 'inst'])
    _presigned = [r for r in g10_redirects
                  if r['status'] == 302 and r['location_is_amazonaws']
                  and r['location_has_signature']]
    record_json('S-10b-download-redirect.json',
                {'redirects': g10_redirects,
                 'presigned_302': len(_presigned)}, quiet=False)
    b_record('S-10b-2', _inst_ok and bool(_presigned),
             f'機関ストレージ一覧/DL {"OK" if _inst_ok else "NG"} / '
             f'presigned への 302 {len(_presigned)} 件')

In [ ]:
if phase == 'b':
    # S-10b-6: intra move → copy。移動して戻す。
    # 成功の知らせは growl ではなく treebeard の行 notify で出る
    # (回帰 NB セル [59] と同じ理由)。表示は 1 秒で消えるので待ち方を変えない。
    async def drag_and_drop_with_alt(page, source, dest):
        """Alt を押しながらドラッグする(= fangorn の copyMode を 'copy' にする)。

        fangorn.js は `$(document).keydown` で `e.altKey` を見て module 変数
        `altKey` を立てる。mouse.up のあとに離すこと。
        """
        await page.keyboard.down('Alt')
        try:
            await grdm.drag_and_drop(page, source, dest)
        finally:
            await page.keyboard.up('Alt')

    async def wait_file_op_notified(page, word):
        try:
            await expect(page.locator(
                f'//*[contains(text(), "{word}に成功しました")]')
            ).to_be_visible(timeout=transition_timeout)
            return True
        except Exception as e:
            print(f'[!] 「{word}に成功しました」を捕まえられなかった: {e!r}')
            return False

    b_intra = {}

    async def first_row(locator, label):
        """同名の行が複数あっても止まらないよう先頭に絞る。

        前の run の intra 試行のコピーが残っていると、同じ名前の行が
        ルート直下とフォルダの中の 2 つ見える。strict mode で落ちると
        move も copy も一度も試せずに終わるので、**件数を記録してから
        先頭を使う**。件数は S-10b-intra.json と判定の文言に残る。
        """
        count = await locator.count()
        b_intra.setdefault('row_counts', {})[label] = count
        if count != 1:
            print(f'[!] 「{label}」に一致する行が {count} 件。先頭を使う')
        return locator.first

    async def _step(page):
        await open_project_files(page, p_addon)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        await clear_growls(page)
        # 移動: 素のドラッグ。対象は phase a で置いた記号入りの名前。
        _row = await first_row(
            grdm.get_select_file_extension_locator(page, f_symbols),
            f'move/file:{f_symbols}')
        await _row.click(timeout=transition_timeout)
        src = await first_row(
            grdm.get_select_file_draggable_locator(page, f_symbols),
            f'move/drag:{f_symbols}')
        dst = await first_row(
            grdm.get_select_folder_droppable_locator(page, folder_top),
            f'move/folder:{folder_top}')
        await grdm.drag_and_drop(page, src, dst)
        b_intra['move'] = {'notified': await wait_file_op_notified(page, '移動')}
        await asyncio.sleep(3)
        b_intra['move'].update({
            'growls': await growl_messages(page),
            'wb': [e for e in wb_responses if e['method'] == 'POST']})
        # コピー: Alt + ドラッグで元の場所に戻す
        wb_responses.clear()
        await clear_growls(page)
        _row = await first_row(
            grdm.get_select_folder_title_locator(page, folder_top),
            f'copy/folder-title:{folder_top}')
        await _row.click(timeout=transition_timeout)
        await asyncio.sleep(2)
        _row = await first_row(
            grdm.get_select_file_extension_locator(page, f_symbols),
            f'copy/file:{f_symbols}')
        await _row.click(timeout=transition_timeout)
        src = await first_row(
            grdm.get_select_file_draggable_locator(page, f_symbols),
            f'copy/drag:{f_symbols}')
        dst = await first_row(
            grdm.get_select_storage_title_locator(page, addon_storage_name),
            f'copy/storage:{addon_storage_name}')
        await drag_and_drop_with_alt(page, src, dst)
        b_intra['copy'] = {'notified': await wait_file_op_notified(page, 'コピー')}
        await asyncio.sleep(3)
        b_intra['copy'].update({
            'growls': await growl_messages(page),
            'wb': [e for e in wb_responses if e['method'] == 'POST']})

    try:
        await run_pw(_step)
    except Exception as _e:
        b_intra['error'] = repr(_e)
        print(f'[!] intra copy/move で例外: {_e!r}(記録して続行)')
    keep_all('S-10b-intra-wb.json')

    # 画面の知らせだけでは足りないので S3 の実体でも確かめる。
    _c = s3_client(s3_region_default)
    for _label, _key in (('moved', f'{folder_top}/{f_symbols}'),
                         ('copied_back', f_symbols)):
        try:
            _h = _c.head_object(Bucket=s3_bucket_mig, Key=_key)
            b_intra[_label] = {'key': _key, 'ContentLength': _h['ContentLength'],
                               'ETag': _h['ETag']}
        except Exception as _e:
            b_intra[_label] = {'key': _key, 'error': repr(_e)}
    record_json('S-10b-intra.json', b_intra, quiet=False)
    _ok = (b_intra.get('move', {}).get('wb')
           and b_intra.get('copy', {}).get('wb')
           and 'error' not in b_intra.get('moved', {})
           and 'error' not in b_intra.get('copied_back', {}))
    _dups = {k: v for k, v in b_intra.get('row_counts', {}).items() if v != 1}
    b_record('S-10b-6', bool(_ok),
             f'move POST {len(b_intra.get("move", {}).get("wb", []))} 本 / '
             f'copy POST {len(b_intra.get("copy", {}).get("wb", []))} 本'
             + (f' / 同名の行が 1 件でないもの {_dups}' if _dups else ''))

In [ ]:
if phase == 'b':
    # S-10b-7: バージョン付きファイルを UI から削除し、S3 側で版が残らないことを見る。
    _before = list_versions(s3_bucket_migver, f_ver)
    record_json('S-10b-versions-before.json', _before, quiet=False)

    # 前の run で削除済みのとき(版が 1 つも無い)。UI から消すものが無いので
    # 削除操作は試さない。試すと「ファイルが無い」で落ちるだけで、何も
    # 確かめられない。再実行では**この項目は確認できない**ことを明示する。
    _already_deleted = _before['version_count'] == 0

    async def _step(page):
        await open_project_files(page, p_ver)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        await delete_file_from_toolbar(page, f_ver)
        # 削除は非同期ではないが、S3 の一覧が追いつくまで少し待つ
        await asyncio.sleep(5)

    if _already_deleted:
        print('[i] 削除前の版が 0。前の run で削除済みとみなして UI 削除は飛ばす')
        b_record('S-10b-7', True,
                 '前の run で削除済み(before=0)。この run では再確認なし')
    else:
        try:
            await run_pw(_step)
            _err = None
        except Exception as _e:
            _err = repr(_e)
            print(f'[!] 削除操作で例外: {_e!r}(記録して続行)')
        keep_all('S-10b-versions-wb.json')

        _after = list_versions(s3_bucket_migver, f_ver)
        record_json('S-10b-versions-after.json', _after, quiet=False)
        b_record('S-10b-7',
                 _after['version_count'] == 0
                 and _after['delete_marker_count'] == 0,
                 f'削除前 版 {_before["version_count"]} / 削除後 版 '
                 f'{_after["version_count"]} + 削除マーカー '
                 f'{_after["delete_marker_count"]}'
                 + (f' / 例外 {_err}' if _err else ''))

In [ ]:
if phase == 'b':
    # S-10b-8: phase c で読む新データを対象 PR 版で置く。
    _new_files = {}
    _p = os.path.join(work_dir, f_new_japanese)
    with open(_p, 'w') as _f:
        _f.write(f'S-10b new 日本語\nrun_id: {run_id}\n')
    _new_files[f_new_japanese] = _p

    if run_chunked:
        _p = os.path.join(work_dir, f_new_chunked)
        with open(_p, 'wb') as _f:
            _remain = chunked_file_size
            while _remain > 0:
                _n = min(_remain, 4 * 1024 * 1024)
                _f.write(os.urandom(_n))
                _remain -= _n
        _new_files[f_new_chunked] = _p

    _new_ver_path = os.path.join(work_dir, f_new_ver)

    async def _step(page):
        await open_project_files(page, p_addon)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        for _n, _path in _new_files.items():
            await upload_and_wait(page, _path)
        # バージョン付きバケットに別名で 3 版
        await open_project_files(page, p_ver)
        await open_storage(page, addon_storage_name)
        for _i in (1, 2, 3):
            with open(_new_ver_path, 'w') as _f:
                _f.write(f'S-10b version {_i}\nrun_id: {run_id}\n')
                _f.write('x' * (100 * _i))
            await upload_overwrite_and_wait(page, _new_ver_path)

    await run_pw(_step)
    keep_all('S-10b-new-upload.json')

    for _n, _path in _new_files.items():
        state['files'][f'addon/{_n}'] = {
            'where': 'addon', 'name': _n, 'placed_in_phase': 'b',
            'sha256': sha256_file(_path), 'size': os.path.getsize(_path)}
    state['files'][f'ver/{f_new_ver}'] = {
        'where': 'ver', 'name': f_new_ver, 'placed_in_phase': 'b',
        'sha256': sha256_file(_new_ver_path),
        'size': os.path.getsize(_new_ver_path)}
    _nv = list_versions(s3_bucket_migver, f_new_ver)
    state['files'][f'ver/{f_new_ver}']['version_count'] = _nv['version_count']
    record_json('S-10b-new-versions.json', _nv, quiet=False)

    # 置いた直後のメタデータも控える(phase c がこれと比べる)。
    async def _step2(page):
        global _new_meta, _new_down
        await open_project_files(page, p_addon)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        _root = await wb_provider_root(page, addon_storage_id)
        _new_meta = await collect_metadata(page, _root)
        _new_down = await collect_downloads(page, _root)

    _new_meta, _new_down = {}, {}
    await run_pw(_step2)
    keep_all('S-10b-new-fetch.json')
    record_json('S-10b-new-metadata.json', _new_meta)
    record_json('S-10b-new-downloads.json', _new_down)
    state['metadata_b_new'] = _new_meta
    state['downloads_b_new'] = _new_down
    save_state(state)
    # _new_down のキーは WB の materialized パスなので basename で引き当てる。
    _new_down_by_name = {os.path.basename(k.rstrip('/')): v
                         for k, v in _new_down.items()}
    _new_by_name = set(_new_down_by_name)
    _not_placed = [n for n in _new_files if n not in _new_by_name]

    # 新データの突合は**この run の投入値**で行う。控えに残っている
    # sha256 / size は前の run の投入値で、置き直す前の中身とは一致しない。
    # (b-3/b-4/b-5 はこれらのキーを対象外にしている。)
    _new_check = {'expected': {}, 'after': {}, 'mismatch': []}
    for _n, _path in sorted(_new_files.items()):
        _want = {'sha256': sha256_file(_path), 'size': os.path.getsize(_path)}
        _new_check['expected'][_n] = _want
        _got = _new_down_by_name.get(_n)
        _new_check['after'][_n] = _got
        if not _got or _got.get('error') or _got.get('status') != 200:
            _new_check['mismatch'].append(
                {'name': _n, 'problem': 'not-downloadable', 'after': _got})
        elif _got.get('sha256') != _want['sha256']:
            _new_check['mismatch'].append(
                {'name': _n, 'problem': 'sha256',
                 'expected': _want['sha256'], 'after': _got.get('sha256')})
        elif _got.get('size') != _want['size']:
            _new_check['mismatch'].append(
                {'name': _n, 'problem': 'size',
                 'expected': _want['size'], 'after': _got.get('size')})

    # 別名の版は別プロジェクトなので、UI を増やさず S3 の実体で確かめる。
    # 小さいファイルなので ETag = md5(マルチパートなら '-' が付くので飛ばす)。
    _ver_want = {'size': os.path.getsize(_new_ver_path),
                 'md5': hashlib.md5(open(_new_ver_path, 'rb').read()).hexdigest()}
    _new_check['expected'][f_new_ver] = _ver_want
    try:
        _vh = s3_client(s3_region_default).head_object(
            Bucket=s3_bucket_migver, Key=f_new_ver)
        _ver_got = {'size': _vh['ContentLength'], 'md5': _vh['ETag'].strip('"')}
    except Exception as _e:
        _ver_got = {'error': repr(_e)}
    _new_check['after'][f_new_ver] = _ver_got
    if _ver_got.get('error'):
        _new_check['mismatch'].append(
            {'name': f_new_ver, 'problem': 's3-head', 'after': _ver_got})
    elif _ver_got['size'] != _ver_want['size']:
        _new_check['mismatch'].append(
            {'name': f_new_ver, 'problem': 'size',
             'expected': _ver_want['size'], 'after': _ver_got['size']})
    elif '-' not in _ver_got['md5'] and _ver_got['md5'] != _ver_want['md5']:
        _new_check['mismatch'].append(
            {'name': f_new_ver, 'problem': 'md5',
             'expected': _ver_want['md5'], 'after': _ver_got['md5']})
    record_json('S-10b-new-check.json', _new_check, quiet=False)

    b_record('S-10b-8',
             not _not_placed and not _new_check['mismatch']
             and _nv['version_count'] >= 2,
             f'新データ {len(_new_files)} 件中 置けなかったもの '
             f'{len(_not_placed)} 件 {_not_placed} / 別名の版 '
             f'{_nv["version_count"]} 件 / この run の投入値との不一致 '
             f'{len(_new_check["mismatch"])} 件')

In [ ]:
if phase == 'b' and run_dotted and (state.get('dot') or {}).get('ok'):
    # S-10b-10(任意): ドット入りバケット。失敗しても判定を落とさない。
    _dot_b = {}
    try:
        async def _step(page):
            await open_project_files(page, p_dot)
            wb_responses.clear()
            await open_storage(page, addon_storage_name)
            _root = await wb_provider_root(page, addon_storage_id)
            _dot_b['metadata'] = await collect_metadata(page, _root)
            _dot_b['downloads'] = await collect_downloads(page, _root)

        await run_pw(_step)
        _dot_b['ok'] = True
    except Exception as _e:
        _dot_b['ok'] = False
        _dot_b['error'] = repr(_e)
        print(f'[!] ドット入りバケットの一覧/DL が失敗した: {_e!r}')
    keep_all('S-10b-dot-wb.json')
    record_json('S-10b-dot.json', _dot_b, quiet=False)
    print('S-10b-10 は参考扱い(判定には入れない)')
elif phase == 'b':
    print('S-10b-10: 比較対象なし(phase a でドット入りバケットを作っていないか失敗した)')

In [ ]:
if phase == 'b':
    # S-10b-11: 新規プロジェクトに**対象 PR 版で新規接続**し、1 件だけ置く。
    # ここは「新実装で新しく接続したものがどう保存されるか」を見るためのもので、
    # phase a が作った設定には触らない。
    _newconn = {'project': p_newconn, 'bucket': s3_bucket_mig}
    _nc_path = os.path.join(work_dir, f_newconn)
    with open(_nc_path, 'w') as _f:
        _f.write(f'S-10b newconn\nrun_id: {run_id}\n')

    async def _step(page):
        await create_project_and_open_files(page, p_newconn)
        # 再実行では既にアカウントが接続済みのことがある。そのときは
        # **バケットを選び直さない**(reselect=False)。選び直すと保存済みの
        # folder_id を今の実装で書き直してしまい、前の run が新規接続した
        # ときの保存形が確かめられなくなる。表示されている folder_id を
        # 読み取って、あとで期待バケットと突き合わせる。
        _conn = await connect_s3_addon(page, p_newconn, s3_bucket_mig,
                                       reselect=False)
        if isinstance(_conn, dict):
            _newconn.update({'connected_via': _conn.get('entered_by'),
                             'folder_id': _conn.get('folder_id'),
                             'reselected': False})
        else:
            _newconn.update({'connected_via': _conn, 'reselected': True})
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        await upload_and_wait(page, _nc_path)
        # 保存された folder_id の形を画面から読む。アドオン設定ページの
        # 選択中バケットの表示が `<bucket>:/` の形かどうかを見る。
        await page.locator('//a[text() = "アドオン"]').click()
        await asyncio.sleep(3)
        _scope = page.locator(f'//*[@id = "{addon_storage_id}Scope"]')
        await expect(_scope).to_be_visible(timeout=transition_timeout)
        _newconn['addon_page_text'] = (await _scope.inner_text())[:2000]
        await page.screenshot(path=evidence('S-10b-newconn-addon.png'),
                              full_page=True)

    try:
        await run_pw(_step)
        _newconn['ok'] = True
    except Exception as _e:
        _newconn['ok'] = False
        _newconn['error'] = repr(_e)
        print(f'[!] 新規接続で例外: {_e!r}(記録して続行)')
    keep_all('S-10b-newconn-wb.json')

    # S3 側に実体が届いたか。
    try:
        _h = s3_client(s3_region_default).head_object(
            Bucket=s3_bucket_mig, Key=f_newconn)
        _newconn['s3'] = {'ContentLength': _h['ContentLength'],
                          'ETag': _h['ETag']}
    except Exception as _e:
        _newconn['s3'] = {'error': repr(_e)}
    _newconn['sha256'] = sha256_file(_nc_path)
    # 接続済みの分岐でも folder_id は確かめる。画面の表示は `<bucket>` の形でも
    # `<bucket>:/` の形でもあり得るので、どちらでも一致とみなす。
    _want_forms = [s3_bucket_mig, f'{s3_bucket_mig}:/']
    _newconn['folder_id_expected'] = _want_forms
    _newconn['folder_id_ok'] = _newconn.get('folder_id') in _want_forms
    record_json('S-10b-newconn.json', _newconn, quiet=False)
    state['newconn'] = _newconn
    state['files'][f'newconn/{f_newconn}'] = {
        'where': 'newconn', 'name': f_newconn, 'placed_in_phase': 'b',
        'sha256': _newconn['sha256'], 'size': os.path.getsize(_nc_path)}
    save_state(state)
    _via = _newconn.get('connected_via')
    _fid, _fid_ok = _newconn.get('folder_id'), _newconn['folder_id_ok']
    if _via == 'already-connected':
        # 再接続はしていないので、確かめられるのは「保存済みの folder_id が
        # 期待どおりか」だけ。一致しなければ記録に残して PASS にはしない。
        b_record('S-10b-11', bool(_newconn.get('ok')) and _fid_ok,
                 '接続済み(前の run で新規接続 PASS)。この run では再接続なし。'
                 'folder_id 一致'
                 if (_newconn.get('ok') and _fid_ok) else
                 f'接続済みだが確かめられない / 操作 '
                 f'{"成功" if _newconn.get("ok") else "失敗"} / '
                 f'folder_id {_fid!r}(期待 {_want_forms}) / '
                 f'S3 側 {_newconn.get("s3")}')
    else:
        b_record('S-10b-11', bool(_newconn.get('ok')),
                 f'新規接続 {"成功" if _newconn.get("ok") else "失敗"} / '
                 f'入り口 {_via} / S3 側 {_newconn.get("s3")}')
    print('`folder_id` の保存形は S-10b-newconn.json の addon_page_text と、'
          '管理 API の node settings で確認すること')

In [ ]:
if phase == 'b':
    # S-10b-9: 全段を通した WB 応答。
    _bad = wb_failures(wb_all)
    record_json('S-10b-wb-all.json', json.loads(_redact(
        json.dumps(wb_all, ensure_ascii=False))))
    b_record('S-10b-9', not _bad, f'4xx/5xx・署名系の文言 {len(_bad)} 件')
    for _e in _bad[:20]:
        print(f'  - {_e["status"]} {_e["method"]} {_e["url"]}')

    # まとめ。**ここで初めて assert する。**
    _lines = ['| 番号 | 結果 | 内容 |', '|---|---|---|']
    for _num in sorted(b_results):
        _r = b_results[_num]
        _lines.append(f'| {_num} | {"PASS" if _r["ok"] else "**FAIL**"} | '
                      f'{_r["detail"]} |')
    _summary = '\n'.join(_lines)
    record('S-10b-summary.md',
           f'# S-10b 判定(run_id {run_id})\n\n{_summary}\n')
    record('S-10b-verdict.txt', _summary)
    save_state(state)
    print()
    print('=' * 74)
    print(' phase b 完了。次にすること')
    print('=' * 74)
    print(' 1. 試験環境を **develop 版** に戻す')
    print(' 2. このノートブックを phase="c" で実行する')
    print(f'    state_path="{state_path}" を必ず渡すこと')
    print('=' * 74)
    _failed = [n for n, r in b_results.items() if not r['ok']]
    assert not _failed, f'S-10b で FAIL がある: {_failed}(S-10b-summary.md を見ること)'

## phase c: ロールバック(**試験環境は develop 版に戻してある**)

phase b が対象 PR 版で置いた新データを、旧実装で読めるかを見る。
終わったら後始末までこの段で行う。

| 番号 | 見るもの | 期待 |
|---|---|---|
| S-10c-1 | `S-10b-new-*` の一覧・ダウンロード・別名 3 版の削除 | 成功 |
| S-10c-2 | S-10b-11 で**新規接続した**プロジェクトを開く | **失敗する見込み。何が起きるかを記録するのが目的** |

### S-10c-2 を「失敗してもよい」扱いにする理由

対象 PR 版はバケットのルートを選んでも接続先を `<バケット名>:/` の形で保存する。
develop 版はこの文字列をそのままバケット名として使うため、ロールバック後に
`<バケット名>:/` という名前のバケットを探しにいく。**この段では例外で止めず、
WB の応答と画面の表示を記録する**。記録は「ロールバックするときは新実装で
接続したアドオンの再接続が要る」という運用手順の根拠にする。

In [ ]:
if phase == 'c':
    c_results = {}

    def c_record(num, ok, detail):
        c_results[num] = {'ok': bool(ok), 'detail': detail}
        print(f'{num}: {"PASS" if ok else "FAIL"} — {detail}')
        return ok

    # S-10c-1: phase b が置いた新データを develop 版で読む。
    c_meta, c_down = {}, {}

    async def _step(page):
        await open_project_files(page, p_addon)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        _root = await wb_provider_root(page, addon_storage_id)
        c_meta['addon'] = await collect_metadata(page, _root)
        c_down['addon'] = await collect_downloads(page, _root)

    await run_pw(_step)
    keep_all('S-10c-fetch.json')
    record_json('S-10c-metadata.json', c_meta)
    record_json('S-10c-downloads.json', c_down)

    _expected = {k: v for k, v in state.get('files', {}).items()
                 if v.get('placed_in_phase') == 'b' and v['where'] == 'addon'}
    # collect_downloads のキーは WB の materialized パス(例 `/S-10b-new-日本語.txt`)。
    # 控えはファイル名で持っているので、basename で引き当てる。
    _by_name = {os.path.basename(k.rstrip('/')): v
                for k, v in (c_down.get('addon') or {}).items()}
    _mismatch = []
    for _key, _want in _expected.items():
        _name = _want['name']
        _got = _by_name.get(_name)
        if _got is None:
            _mismatch.append({'name': _name, 'problem': 'not-downloadable',
                              'available': sorted(_by_name)})
        elif _got['sha256'] != _want['sha256']:
            _mismatch.append({'name': _name, 'expected': _want['sha256'],
                              'actual': _got['sha256']})
    record_json('S-10c-downloads-diff.json', _mismatch, quiet=False)
    c_record('S-10c-1-downloads', not _mismatch,
             f'新データ {len(_expected)} 件中 不一致 {len(_mismatch)} 件')

In [ ]:
if phase == 'c':
    # S-10c-1(続き): phase b が作った別名 3 版を develop 版の UI から削除する。
    _before = list_versions(s3_bucket_migver, f_new_ver)
    record_json('S-10c-versions-before.json', _before, quiet=False)

    async def _step(page):
        await open_project_files(page, p_ver)
        wb_responses.clear()
        await open_storage(page, addon_storage_name)
        await delete_file_from_toolbar(page, f_new_ver)
        await asyncio.sleep(5)

    try:
        await run_pw(_step)
        _err = None
    except Exception as _e:
        _err = repr(_e)
        print(f'[!] 削除操作で例外: {_e!r}(記録して続行)')
    keep_all('S-10c-versions-wb.json')

    _after = list_versions(s3_bucket_migver, f_new_ver)
    record_json('S-10c-versions-after.json', _after, quiet=False)
    c_record('S-10c-1-versions',
             _after['version_count'] == 0 and _after['delete_marker_count'] == 0,
             f'削除前 版 {_before["version_count"]} / 削除後 版 '
             f'{_after["version_count"]} + 削除マーカー '
             f'{_after["delete_marker_count"]}'
             + (f' / 例外 {_err}' if _err else ''))

In [ ]:
if phase == 'c':
    # S-10c-2: 対象 PR 版で新規接続したプロジェクトを develop 版で開く。
    # **失敗が期待値。例外で止めず、何が起きたかを残す。**
    _c2 = {'project': p_newconn,
           'note': ('対象 PR 版は接続先を `<バケット名>:/` の形で保存する。'
                    'develop 版はこれをそのままバケット名として使うので、'
                    'ここは失敗する見込み。失敗の仕方を記録するのが目的。')}

    async def _step(page):
        await open_project_files(page, p_newconn)
        wb_responses.clear()
        try:
            await open_storage(page, addon_storage_name)
            _c2['opened'] = True
        except Exception as e:
            _c2['opened'] = False
            _c2['open_error'] = repr(e)
        await asyncio.sleep(5)
        _c2['growls'] = await growl_messages(page)
        _c2['tree_text'] = (await page.locator(
            '//*[@id = "treeGrid"]').inner_text())[:2000]
        await page.screenshot(path=evidence('S-10c-newconn.png'),
                              full_page=True)

    try:
        await run_pw(_step)
    except Exception as _e:
        _c2['error'] = repr(_e)
        print(f'[!] S-10c-2 で例外: {_e!r}(これ自体が記録対象。続行する)')

    _c2['wb'] = [{'status': e['status'], 'method': e['method'],
                  'url': e['url'], 'body': (e.get('body') or '')[:1000]}
                 for e in wb_responses]
    _c2['wb_failures'] = [{'status': e['status'], 'url': e['url']}
                          for e in wb_failures()]
    keep_all('S-10c-newconn-wb.json')
    record_json('S-10c-newconn.json', json.loads(_redact(
        json.dumps(_c2, ensure_ascii=False))), quiet=False)
    # **判定には入れない。** 事実の記録だけ。
    print()
    print('S-10c-2 は判定項目ではない(失敗が期待値)。'
          'S-10c-newconn.json の内容を運用手順の根拠にすること')

In [ ]:
if phase == 'c':
    # S-10c のまとめ。
    _bad = wb_failures(wb_all)
    _lines = ['| 番号 | 結果 | 内容 |', '|---|---|---|']
    for _num in sorted(c_results):
        _r = c_results[_num]
        _lines.append(f'| {_num} | {"PASS" if _r["ok"] else "**FAIL**"} | '
                      f'{_r["detail"]} |')
    _lines.append(f'| S-10c-2 | 記録のみ | 判定には入れない'
                  f'(S-10c-newconn.json)|')
    record('S-10c-summary.md',
           f'# S-10c 判定(run_id {run_id})\n\n' + '\n'.join(_lines)
           + f'\n\nWB 応答の 4xx/5xx・署名系の文言(S-10c-2 を含む全段): '
             f'{len(_bad)} 件\n')
    record('S-10c-verdict.txt', '\n'.join(_lines))
    record_json('S-10c-wb-all.json', json.loads(_redact(
        json.dumps(wb_all, ensure_ascii=False))))

In [ ]:
if phase == 'c':
    # 後始末(回帰 NB セル [71] の複製)。
    # - パラメータで**指定された**バケット(ユーザー所有)は、本 NB が置いた
    #   残骸だけを消す。バケット自体は消さない。
    # - phase a が**生成した**バケットは中身ごと delete_bucket する。
    _generated_names = {n for n, r in generated_bucket_names}
    _leftovers = []


    def _delete_keys(bucket, keys, region):
        """指定したキーを全バージョン消す(バージョニング有効でも残さない)。"""
        c = s3_client(region)
        deleted = 0
        for key in keys:
            while True:
                r = c.list_object_versions(Bucket=bucket, Prefix=key)
                objs = [{'Key': v['Key'], 'VersionId': v['VersionId']}
                        for v in r.get('Versions', []) + r.get('DeleteMarkers', [])
                        if v['Key'] == key or v['Key'].startswith(key)]
                if not objs:
                    break
                for i in range(0, len(objs), 1000):
                    c.delete_objects(Bucket=bucket,
                                     Delete={'Objects': objs[i:i + 1000]})
                deleted += len(objs)
        return deleted


    # 本 NB が置いたキー。S3 上のキーは materialized パスの先頭の / を外したもの。
    _placed = sorted({
        (v.get('key') or ((v.get('folder') + '/') if v.get('folder') else '')
         + v['name'])
        for v in state.get('files', {}).values()})
    _placed += [f'{folder_top}/', 'S-10-raw/']
    for _role, _bucket in (('mig', s3_bucket_mig), ('migver', s3_bucket_migver),
                           ('dot', s3_bucket_dot)):
        if not _bucket:
            continue
        if _bucket in _generated_names:
            _leftovers.append(f'  {_bucket}: 生成バケットなので残骸削除は省略'
                              '(このあと delete_bucket する)')
            continue
        _n = _delete_keys(_bucket, _placed, s3_region_default)
        _left = s3_client(s3_region_default).list_objects_v2(
            Bucket=_bucket, MaxKeys=20)
        _leftovers.append(
            f'  {_bucket}(指定バケット): {_n} 件削除 / 残り KeyCount='
            f'{_left.get("KeyCount", 0)} '
            f'{[o["Key"] for o in _left.get("Contents", [])]}')

    # **ここでだけ True にする。**
    delete_generated_buckets = True
    _text = cleanup_buckets(verbose=True)
    record('S-10c-cleanup.txt',
           '--- 指定バケット(ユーザー所有)の残骸削除 ---\n'
           + ('\n'.join(_leftovers) or '  対象なし')
           + '\n\n--- 生成バケットの削除 ---\n' + _text + '\n')
    _all = [b['Name'] for b in
            s3_client(s3_region_default).list_buckets().get('Buckets', [])]
    _made = [n for n, r in generated_bucket_names]
    _left = [b for b in _made if b in _all]
    print(f'生成 {len(_made)} / 削除 {len(_made) - len(_left)} / 残存 {len(_left)}')
    for _b in _made:
        print(f'  - {_b}' + ('  **残存**' if _b in _left else ''))
    assert not _left, (
        f'生成したバケットが残っている: {_left}。'
        'S-10c-cleanup.txt を見て手で削除すること')

### 後始末チェックリスト(phase c)

**この試験は機関ストレージの設定を 1 回上書きしている。**
`S-10a-institutional-storage-before.txt` に登録前の状態が残っているので、
それを見ながら次を消化すること。

- [ ] 機関ストレージを `S-10a-institutional-storage-before.txt` の状態に戻す
      (**手動**。管理画面で元のプロバイダを選び直して保存する。
      「登録を外す」操作は無い)
- [ ] 本ノートブックが作ったプロジェクトを削除する(次のセルが名前を出す)
- [ ] ユーザー設定 →「アドオンアカウント構成」から S3 アカウントの連携を解除する
      (アドオンのアカウントは**ユーザー単位**で残る)
- [ ] 生成したバケットが消えていることを確認する(上のセル)
- [ ] `S-10-state.json` を証跡としてまとめて保管する

In [ ]:
if phase == 'c':
    # 削除対象のプロジェクト(削除は手動で行う)。
    print('削除対象のプロジェクト:')
    for _p in [p_inst, p_addon, p_ver, p_dot, p_newconn]:
        if _p:
            print(' -', _p)
    print()
    print('RDM-e2e-test-nb の scripts/cleanup_sigv4_run.py が'
          'プロジェクト削除とアドオン連携解除を行う:')
    print(f'  python3 scripts/cleanup_sigv4_run.py --prefix {rdm_project_prefix}'
          ' --dry-run')
    print('  (--prefix は本 NB のプロジェクト名の接頭辞。接尾辞の組が違うので、'
          '対象が出ないときは手で削除すること)')

In [ ]:
# 証跡の INDEX。
_scenario = {
    'S-10a': '仕込み(develop 版)',
    'S-10b': '検証(対象 PR 版)',
    'S-10c': 'ロールバック(develop 版)',
    'S-10-': '段をまたぐ控え',
}


def write_index():
    """run_dir の中身から INDEX.md を作る。"""
    _index = ['# 証跡 INDEX (S-10 移行互換)',
              '',
              f'- phase: {phase}',
              f'- run_id: {run_id}',
              f'- 取得日時: {datetime.now().isoformat()}',
              f'- 対象: RCOSDP/RDM-waterbutler#93 / RCOSDP/RDM-osf.io#746',
              f'- 控え: {state_path}',
              '',
              '| ファイル | サイズ | 段 |',
              '|---|---|---|']
    for _f in sorted(os.listdir(run_dir)):
        if _f == 'INDEX.md' or _f.startswith(('har', 'video-', 'last-')) \
                or _f == 'console.log':
            continue
        _label = next((v for k, v in _scenario.items() if _f.startswith(k)),
                      'Playwright 成果物')
        _index.append(f'| `{_f}` | '
                      f'{os.path.getsize(os.path.join(run_dir, _f))} | '
                      f'{_label} |')
    _index.append('')
    _index.append('`har.zip` / `video-*.webm` / `console.log` / '
                  '`last-*.{png,html}` は `scripts/playwright.py` が '
                  '`finish_pw_context` で書き出したもの。')
    record('INDEX.md', '\n'.join(_index), quiet=True)


write_index()

In [ ]:
# 回帰 NB セル [68] の複製。
# Playwright を閉じる。**これをしないと HAR が書き出されない。**
# 既に har.zip があるなら退避する(finish_pw_context は同じ場所に上書きする)。
_har = os.path.join(run_dir, 'har.zip')
if os.path.exists(_har):
    _n = 1
    while os.path.exists(os.path.join(run_dir, f'har-{_n}.zip')):
        _n += 1
    shutil.copyfile(_har, os.path.join(run_dir, f'har-{_n}.zip'))
    print(f'既存の HAR を har-{_n}.zip に退避した')

await finish_pw_context(timeout=300)

In [ ]:
# 回帰 NB セル [70] の複製。HAR から presigned への 302 を数える。
# **phase b 以外では参考値**(302 が 0 でも落とさない)。対象 PR 版で
# 302 が出ないことだけが欠陥の兆候で、develop 版の段はその判定の場ではない。
import zipfile

_har_zip = os.path.join(run_dir, 'har.zip')
har_verdict = {'phase': phase, 'har_zip': _har_zip,
               'exists': os.path.exists(_har_zip)}

if har_verdict['exists']:
    with zipfile.ZipFile(_har_zip) as _z:
        _member = [n for n in _z.namelist() if n.endswith('.har')][0]
        _har = json.loads(_z.read(_member).decode('utf-8'))

    _redirects = []
    for _e in _har['log']['entries']:
        _req, _res = _e['request'], _e['response']
        if _req['method'] != 'GET' or '/v1/resources/' not in _req['url']:
            continue
        if not re.search(r'/providers/(osfstorage|s3)/', _req['url']):
            continue
        # `revisions=` は版一覧でダウンロードではない。
        if ('meta=' in _req['url'] or 'zip=' in _req['url']
                or 'revisions=' in _req['url']):
            continue
        _loc = ''
        for _h in _res.get('headers', []):
            if _h['name'].lower() == 'location':
                _loc = _h['value']
        _redirects.append(redirect_entry(_req['url'], _res['status'], _loc))

    _ok = [r for r in _redirects + g10_redirects
           if r['status'] == 302 and r['location_is_amazonaws']
           and r['location_has_signature']]
    har_verdict['entries'] = len(_har['log']['entries'])
    har_verdict['download_candidates'] = _redirects
    har_verdict['live_redirects'] = g10_redirects
    har_verdict['presigned_redirects'] = len(_ok)
    record(f'S-10{phase}-download-redirect.json',
           _redact(json.dumps(har_verdict, indent=2, ensure_ascii=False)))
    print(f'presigned URL への 302: {len(_ok)} 件')
    if phase == 'b' and not _ok:
        # 「経路を踏んでいない」と「踏んだのに 302 にならない」は結論が全く違う。
        # 取り違えると欠陥を捏造することになるので言い分ける。
        _nd = [r for r in _redirects + g10_redirects if not r.get('has_direct')]
        print('[!] phase b で presigned への 302 が 1 件も無い。'
              + ('ダウンロード要求がすべて `direct` 付きで、その経路を'
                 '踏んでいない。一覧のツールバーの［ダウンロード］で'
                 '取り直すこと'
                 if not _nd else
                 '`direct` の無いダウンロード要求はあるのに 302 に'
                 'なっていない')
              + f'。S-10{phase}-download-redirect.json を見ること')
else:
    record_json(f'S-10{phase}-download-redirect.json', har_verdict, quiet=False)
    print(f'[!] {_har_zip} が無い。default_result_path が run_dir を指していないか、'
          'finish_pw_context が走っていない')

write_index()

In [ ]:
print('証跡:', run_dir)
print('控え:', state_path)
print('作業ディレクトリを削除します:', work_dir)

In [ ]:
!rm -fr {work_dir}